<div style="background-color: #F4F6F7; padding: 20px; border-radius: 8px; font-family: 'Arial', sans-serif; color: #2E4053;">
  <!-- Logo centrado -->
  <div style="text-align: center; margin-bottom: 15px;">
    <img src="https://drive.google.com/uc?export=view&id=1kjzXfjTiieYAd4azw5bh4UfEg91gUIdh" alt="Logo Institucional" width="250" />
  </div>

  <!-- Título principal -->
  <h1 style="font-size: 36px; font-weight: bold; text-align: right;">
    <strong>Estimación del gradiente geotérmico a partir de imágenes multiespectrales del LANDSAT 7 y base de datos tabular mediante un sistema con atención intramodal e intermodal residual, además de weighted sampler, selección de variables y clipping de atípicos</strong>
  </h1>

  <!-- Datos del estudiante -->
  <p style="font-size: 22px; text-align: center; margin: 5px 0;">
    <strong>Nombre:</strong> Carlos Enrique Montilla Morales &nbsp;|&nbsp;
    <strong>Institución:</strong> Universidad Tecnológica de Pereira  &nbsp;|&nbsp;
    <strong>Grupo de investigación en Automática </strong>
  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />

  <!-- Descripción breve -->
  <p style="font-size: 18px; text-align: center; font-style: italic; margin: 5px 0;">
    Este cuaderno es la primera prueba donde se entrena un modelo multimodal mediante el uso de imágenes satelitales multiespectrales del LANDSAT7 y una base de datos que contienen diversas variables geológicas y geofísicas para estimar el gradiente geotérmico en Colombia. Incluye: ingestión y preprocesado de mosaicos (manejo de NoData), creación de muestras/parches, estrategia de fine-tuning, evaluación con métricas de regresión R2, MAE y RMSE y exportación de predicciones y checkpoints para análisis geoespacial.

  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />
</div>


# **1. Carga de las librerías, imágenes y la base de datos**

In [ ]:
!pip install -q rasterio==1.4 pyproj rioxarray
!pip install -q torchvision tqdm scikit-image pandas
!pip install -q torch-optimizer optuna
!pip install -q einops huggingface_hub universal-pathlib
!pip install -q --no-deps olmoearth-pretrain-minimal

import warnings
warnings.filterwarnings(
    "ignore",
    message="This overload of addcmul_ is deprecated"
)


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
!gdown 1OIJY8QtxRYr4HW504Awxama22OAuBJID # Base de datos cruda
#!gdown 1oZAsFy_yDXnJUbUQbuIMlIS-SFaEhd_X #Geotiffs completos
#!gdown 1CnCNvGQSoWNEIbNz-j5bIa_COHpK15-l #Geotiffs completos 256
#!gdown 1wfjvuzJcSUH60iDi4wlvAIvOWLcdenKK #Geotiffs completos 224 y 9 bandas
#!gdown 1r2c3X8AlXOReL611kt7FqdRUu18jLsKf #Geotiff completos 224 y 10 bandas
!gdown 1r5wKeI-GGlznDuOljIGXu08egrgfY8UN #Geotiff completos 224 y 10 bandas con knn

!unzip -q LANDSAT_GEOTIFF_10_bands_knn.zip -d LANDSAT_GEOTIFF

In [ ]:
import os 
zip_path = "/kaggle/working/LANDSAT_GEOTIFF_10_bands_knn.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)
    print("ZIP eliminado")
else:
    print("No se encontró el ZIP")

In [ ]:
import os, re, time, joblib, json, shutil, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import rasterio
from scipy.spatial import cKDTree
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch_optimizer as optim
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import NearestNeighbors

import geopandas as gpd
from pyproj import Transformer

BASE_DIR = Path("/kaggle/working/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF")
CSV_BASE = Path("/kaggle/working/data_prep.csv")
TARGET_COL = "Apparent Geothermal Gradient (°C/Km)"
OUT_DIR = Path("/kaggle/working/run_olmoearth_tiny_wsmoter_attention")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OPTUNA_TMP_DIR = Path("/kaggle/temp/optuna_trials")
TOP_K_CKPT_DIR = OUT_DIR / "optuna_top5_checkpoints"
TOP_K = 5

OPTUNA_TMP_DIR.mkdir(parents=True, exist_ok=True)
TOP_K_CKPT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (224, 224)
FILL_METHOD_IMAGE = 'knn'
K_NEIGHBORS = 8
IMAGE_NORM = 'zscore'
TAB_IMPUTE_METHOD = 'knn'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
NUM_WORKERS = 2

EPOCHS_HEAD = 4
EPOCHS_BACKBONE = 4
EPOCHS_FULL = 5

LR_HEAD = 1e-4
LR_FINE = 1e-5
LR_FULL = 1e-6
WEIGHT_DECAY = 1e-3
MAX_GRAD_NORM = 1.0

SEED = 42

# AMP + scheduler fijo
USE_AMP = (DEVICE.type == "cuda")

# Scheduler fuera de Optuna
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2
SCHEDULER_MIN_LR = 1e-8

# Split Conformal Prediction
# 70% train / 10% calibration / 10% validation / 10% test.
CONFORMAL_ALPHA = 0.10          # 90% de cobertura nominal
CONFORMAL_NOMINAL_COVERAGE = 1.0 - CONFORMAL_ALPHA

# WSMOTER tabular
USE_WSMOTER_TABULAR = True
WSMOTER_MULTIPLIER = 1.3      # 1.35 = crea +35% muestras sintéticas en train
WSMOTER_K_NEIGHBORS = 5
WSMOTER_WEIGHT_POWER = 2     # >1 aumenta la probabilidad de generar extremos
WSMOTER_ID_MODE = "anchor"     # La imagen se conserva usando el ID de la muestra ancla
LDS_SIGMA = 5
LDS_N_BINS = 15

# Optuna
USE_OPTUNA = True
OPTUNA_N_TRIALS = 50
OPTUNA_TIMEOUT = None    
OPTUNA_STUDY_NAME = "olmoearth_tiny_multimodal_optuna"
OPTUNA_DIR = OUT_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

# Aumentación espacial para imágenes
USE_IMAGE_AUGMENTATION = True
AUG_P_HFLIP = 0.50
AUG_P_VFLIP = 0.50
AUG_P_ROT90 = 0.50
AUG_P_NOISE = 0.35
AUG_NOISE_SIGMA_IMAGE = 0.025
AUG_NOISE_SIGMA_THERMAL = 0.020
AUG_CLIP_VALUE = 5.0


# 9 bandas: sin Band2.
# 10 bandas: con Band2.
BAND_ORDER_9 = [
    "Band61_TOA_BT",
    "Band1_SRF_REF",
    "Band3_SRF_REF",
    "Band4_SRF_REF",
    "Band5_SRF_REF",
    "Band7_SRF_REF",
    "NDVI_SRF",
    "Band3_div_Band1",
    "Band5_div_Band7"
]

BAND_ORDER_10 = [
    "Band61_TOA_BT",
    "Band1_SRF_REF",
    "Band2_SRF_REF",
    "Band3_SRF_REF",
    "Band4_SRF_REF",
    "Band5_SRF_REF",
    "Band7_SRF_REF",
    "NDVI_SRF",
    "Band3_div_Band1",
    "Band5_div_Band7"
]

def detect_band_config(base_dir):
    tif_files = sorted(Path(base_dir).glob("*.tif"))
    if not tif_files:
        raise FileNotFoundError(f"No se encontraron GeoTIFF en {base_dir}")

    with rasterio.open(tif_files[0]) as ds:
        n_bands = ds.count

    if n_bands == 9:
        band_order = BAND_ORDER_9
        thermal_idxs = [0, 6, 7, 8]      # Band61, NDVI, B3/B1, B5/B7
    elif n_bands == 10:
        band_order = BAND_ORDER_10
        thermal_idxs = [0, 7, 8, 9]      # Band61, NDVI, B3/B1, B5/B7
    else:
        raise ValueError(
            f"Se detectaron {n_bands} bandas. Este notebook espera 9 o 10 bandas."
        )

    return n_bands, band_order, thermal_idxs

N_IMAGE_CHANNELS, BAND_ORDER, THERMAL_DERIVED_IDXS = detect_band_config(BASE_DIR)
IMAGE_IDXS = list(range(N_IMAGE_CHANNELS))
N_THERMAL_DERIVED_CHANNELS = len(THERMAL_DERIVED_IDXS)

# Caché temporal: no se guarda como output permanente de Kaggle.
CACHE_DIR = Path('/kaggle/temp') / f'image_cache_{TARGET_SIZE[0]}x{TARGET_SIZE[1]}_{N_IMAGE_CHANNELS}ch'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Backbone visual: OlmoEarth v1.1 Tiny
OLMOEARTH_MODEL_ID = "allenai/OlmoEarth-v1_1-Tiny"
OLMOEARTH_MODEL_SIZE = "tiny"
PRETRAINED_OLMOEARTH = True
OLMOEARTH_PATCH_SIZE = 8
OLMOEARTH_INPUT_RES = 30       # Landsat 7 SR suele estar a 30 m/píxel.
OLMOEARTH_TARGET_SIZE = TARGET_SIZE
OLMOEARTH_EMBED_DIM = 192
OLMOEARTH_USE_THERMAL_IN_BACKBONE = False

# Orden de bandas que espera la modalidad Landsat de OlmoEarth.
OLMOEARTH_LANDSAT_ORDER = ["B8", "B1", "B2", "B3", "B4", "B5", "B6", "B7", "B9", "B10", "B11"]

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("Device:", DEVICE)
print("USE_WSMOTER_TABULAR =", USE_WSMOTER_TABULAR)
print("USE_IMAGE_AUGMENTATION =", USE_IMAGE_AUGMENTATION)
print("N_IMAGE_CHANNELS =", N_IMAGE_CHANNELS)
print("BAND_ORDER =", BAND_ORDER)
print("THERMAL_DERIVED_IDXS =", THERMAL_DERIVED_IDXS)
print("N_THERMAL_DERIVED_CHANNELS =", N_THERMAL_DERIVED_CHANNELS)
print("USE_AMP =", USE_AMP)
print("CACHE_DIR =", CACHE_DIR)
print("Scheduler fijo:", SCHEDULER_FACTOR, SCHEDULER_PATIENCE, SCHEDULER_MIN_LR)


In [ ]:
path = BASE_DIR / "LANDSAT7_1.tif"   # cambia por un ID real

with rasterio.open(path) as ds:
    print("Número de bandas:", ds.count)
    print("Shape:", ds.height, ds.width)
    print("Descriptions:", ds.descriptions)
    print("Indexes:", ds.indexes)

    for i in ds.indexes:
        print(f"\n--- Banda {i} ---")
        print("Descripción:", ds.descriptions[i-1])
        print("Tags:", ds.tags(i))

In [ ]:
# Cell 2

def read_geotiff(path: Path):
    with rasterio.open(str(path)) as ds:
        arr = ds.read().astype(np.float32)
        nodata = ds.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        return arr, ds.meta

def split_optical_thermal(arr):
    """
    Conserva los nombres optical/thermal para modificar lo mínimo del notebook.

    optical  -> stack completo de N_IMAGE_CHANNELS para la rama visual.
    thermal  -> rama auxiliar con Band61 + NDVI + Band3/Band1 + Band5/Band7.
    """
    if arr.shape[0] < N_IMAGE_CHANNELS:
        raise ValueError(
            f"El GeoTIFF tiene {arr.shape[0]} bandas, pero este modelo espera "
            f"{N_IMAGE_CHANNELS}. Revisa que estés usando los recortes correctos."
        )

    optical = arr[IMAGE_IDXS].astype(np.float32)
    thermal = arr[THERMAL_DERIVED_IDXS].astype(np.float32)

    return optical, thermal

def fill_missing_median(arr):
    out = arr.copy()
    for b in range(out.shape[0]):
        band = out[b]
        mask = np.isnan(band)
        if mask.any():
            med = np.nanmedian(band)
            band[mask] = med if not np.isnan(med) else 0.0
        out[b] = band
    return out

def fill_missing_knn_mean(arr, k=8):
    out = arr.copy()
    for b in range(out.shape[0]):
        band = out[b]
        nan = np.isnan(band)
        if not nan.any():
            continue

        ys, xs = np.where(~nan)

        if len(ys) == 0:
            band[nan] = 0.0
            out[b] = band
            continue

        tree = cKDTree(np.c_[ys, xs])
        ys_n, xs_n = np.where(nan)

        _, idx = tree.query(np.c_[ys_n, xs_n], k=min(k, len(ys)))

        if np.ndim(idx) == 1:
            band[ys_n, xs_n] = band[ys[idx], xs[idx]]
        else:
            band[ys_n, xs_n] = band[ys[idx], xs[idx]].mean(axis=1)

        out[b] = band

    return out

def resize_multiband(arr, target_size=TARGET_SIZE):
    bands, H, W = arr.shape
    Ht, Wt = target_size

    if (H, W) == (Ht, Wt):
        return arr.astype(np.float32)

    arr_t = np.transpose(arr, (1, 2, 0))

    resized = resize(
        arr_t,
        (Ht, Wt, bands),
        order=1,
        preserve_range=True,
        anti_aliasing=True
    )

    return np.transpose(resized, (2, 0, 1)).astype(np.float32)


# Caché de imágenes preprocesadas
def _cache_path_for_id(sample_id):
    return CACHE_DIR / f"LANDSAT7_{int(sample_id)}.npy"


def preprocess_full_image(sample_id):
    """
    Transformaciones deterministas que se cachean:
      1) lectura GeoTIFF
      2) selección de las 9/10 bandas del modelo
      3) relleno de NaN
      4) resize a TARGET_SIZE

    NO aplica normalización ni augmentación.
    """
    arr, _ = read_geotiff(BASE_DIR / f"LANDSAT7_{int(sample_id)}.tif")

    if arr.shape[0] < N_IMAGE_CHANNELS:
        raise ValueError(
            f"LANDSAT7_{int(sample_id)}.tif tiene {arr.shape[0]} bandas; "
            f"se esperaban al menos {N_IMAGE_CHANNELS}."
        )

    image = arr[IMAGE_IDXS].astype(np.float32)
    image = fill_missing_median(image)
    image = resize_multiband(image, TARGET_SIZE)

    # float16 solo en disco para ahorrar espacio.
    return np.ascontiguousarray(image, dtype=np.float16)


def get_cached_full_image(sample_id):
    cache_path = _cache_path_for_id(sample_id)

    if not cache_path.exists():
        image = preprocess_full_image(sample_id)
        tmp_path = cache_path.with_suffix('.tmp.npy')
        np.save(tmp_path, image, allow_pickle=False)
        os.replace(tmp_path, cache_path)

    # El pipeline original trabaja en float32.
    return np.load(cache_path, allow_pickle=False).astype(np.float32, copy=False)


def build_image_cache(df_split, desc='Construyendo caché de imágenes'):
    unique_ids = pd.Series(df_split['ID']).astype(int).drop_duplicates().tolist()

    created = 0
    existing = 0

    for sample_id in tqdm(unique_ids, desc=desc):
        cache_path = _cache_path_for_id(sample_id)

        if cache_path.exists():
            existing += 1
            continue

        image = preprocess_full_image(sample_id)
        tmp_path = cache_path.with_suffix('.tmp.npy')
        np.save(tmp_path, image, allow_pickle=False)
        os.replace(tmp_path, cache_path)
        created += 1

    print(
        f"[CACHE] total={len(unique_ids)} | creadas={created} | "
        f"existentes={existing} | {CACHE_DIR}"
    )

def _update_running_channel_stats(arr, sums, sq_sums, counts):
    """
    Estadística por canal sin concatenar todos los píxeles en memoria.
    arr: [C, H, W]
    """
    for c in range(arr.shape[0]):
        vals = arr[c].reshape(-1)
        vals = vals[np.isfinite(vals)]

        if vals.size == 0:
            continue

        sums[c] += vals.sum(dtype=np.float64)
        sq_sums[c] += np.square(vals, dtype=np.float64).sum(dtype=np.float64)
        counts[c] += vals.size

    return sums, sq_sums, counts

def _finalize_running_stats(sums, sq_sums, counts):
    mean = sums / np.maximum(counts, 1)
    var = (sq_sums / np.maximum(counts, 1)) - mean**2
    var = np.maximum(var, 1e-12)
    std = np.sqrt(var) + 1e-8

    return mean.astype(np.float32), std.astype(np.float32)

def compute_image_stats_train(df_split):
    sums = np.zeros(N_IMAGE_CHANNELS, dtype=np.float64)
    sq_sums = np.zeros(N_IMAGE_CHANNELS, dtype=np.float64)
    counts = np.zeros(N_IMAGE_CHANNELS, dtype=np.float64)

    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"Image stats {N_IMAGE_CHANNELS}ch"):
        image = get_cached_full_image(int(row.ID))
        sums, sq_sums, counts = _update_running_channel_stats(image, sums, sq_sums, counts)

    return _finalize_running_stats(sums, sq_sums, counts)

def compute_thermal_stats_train(df_split):
    sums = np.zeros(N_THERMAL_DERIVED_CHANNELS, dtype=np.float64)
    sq_sums = np.zeros(N_THERMAL_DERIVED_CHANNELS, dtype=np.float64)
    counts = np.zeros(N_THERMAL_DERIVED_CHANNELS, dtype=np.float64)

    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc="Thermal/derived stats"):
        image = get_cached_full_image(int(row.ID))
        thermal = image[THERMAL_DERIVED_IDXS].astype(np.float32, copy=False)
        sums, sq_sums, counts = _update_running_channel_stats(thermal, sums, sq_sums, counts)

    return _finalize_running_stats(sums, sq_sums, counts)


In [ ]:
# Cell 3

df_base = pd.read_csv(CSV_BASE, delimiter=';')
df_base = df_base.drop(columns=['Unnamed: 23'])
# df_base = df_base[['Latitude', 'Longitude', 'Elevation (m)',
#        'Curie Depth (Km)','Moho Depth (m)',
#        'Strike-slip Fault', 'Reverse or Thrust Fault',
#        'Right-lateral Fault', 'Nearest Basement',
#        'Normal Fault', 'Active Fault','Magnetic Anomaly (nT)',
#        'Vertical Gravity Gradient (E)', 'Free Air Anomaly (mGal)',
#        'Bouguer Anomaly (mGal)', 'ID', 'Apparent Geothermal Gradient (°C/Km)']]

df = df_base.copy()
df['ID'] = df['ID'].astype(str).str.extract(r'(\d+)').astype(int)
df['tif_name'] = df['ID'].apply(lambda x: f"LANDSAT7_{x}.tif")
df = df[df['tif_name'].apply(lambda x: (BASE_DIR/x).exists())].reset_index(drop=True)

FINAL_CSV = OUT_DIR / "final_dataset.csv"
df.to_csv(FINAL_CSV, index=False)
print("CSV final:", FINAL_CSV)

In [ ]:
# Cargar datos
df_final = pd.read_csv(FINAL_CSV)

extreme_gradient_threshold = df_final['Apparent Geothermal Gradient (°C/Km)'].quantile(0.99)
print(f"Extreme gradient threshold (97.5th percentile): {extreme_gradient_threshold}")

lowest_gradient_threshold = df_final['Apparent Geothermal Gradient (°C/Km)'].quantile(0.01)
print(f"Lowest gradient threshold (0.25st percentile): {lowest_gradient_threshold}")

df_final = df_final[df_final['Apparent Geothermal Gradient (°C/Km)'] <= extreme_gradient_threshold]
df_final = df_final[df_final['Apparent Geothermal Gradient (°C/Km)'] >= lowest_gradient_threshold]

df_final.info()

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d


def bins_freedman_diaconis(x, n):
    x = np.asarray(x, dtype=float)

    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    iqr = q3 - q1

    h = 2 * iqr / (n ** (1 / 3))

    if (not np.isfinite(h)) or h <= 0:
        return 10

    k = np.ceil((x.max() - x.min()) / h)

    if (not np.isfinite(k)) or k < 2:
        return 2

    return int(k)


def add_gradient_weights_lds(
    df,
    target_col,
    weight_col="Gradient Weight",
    sigma=15,
    n_bins=LDS_N_BINS
):
    """
    Calcula pesos LDS solo para el dataframe recibido.

    Para evitar fuga de información, esta función debe aplicarse al split
    de entrenamiento, no a df_final completo.
    """
    df = df.copy()

    y = df[target_col].to_numpy(dtype=float)

    #n_bins = bins_freedman_diaconis(y, len(y))
    hist, bin_edges = np.histogram(y, bins=n_bins)

    smoothed_hist = gaussian_filter1d(hist.astype(float), sigma=sigma)
    smoothed_hist = np.maximum(smoothed_hist, 1e-8)

    bin_ids = np.digitize(y, bin_edges[:-1]) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    density = smoothed_hist[bin_ids]

    weights = 1.0 / density
    weights = weights / weights.mean()

    df[weight_col] = weights.astype(np.float32)

    print(f"[LDS] Pesos calculados solo sobre train | sigma={sigma} | bins={n_bins}")
    print(f"[LDS] Max weight: {weights.max():.6f}")
    print(f"[LDS] Min weight: {weights.min():.6f}")
    print(f"[LDS] Mean weight: {weights.mean():.6f}")
    print(f"[LDS] Median weight: {np.median(weights):.6f}")

    return df


In [ ]:
# Dataset tabular base.
GEO_LAT_COL = "Latitude_raw"
GEO_LON_COL = "Longitude_raw"
PROJECTED_X_COL = "x_m"
PROJECTED_Y_COL = "y_m"
PROJECTED_CRS = "EPSG:32618"  # WGS84 / UTM zona 18N

df_final = df_final[['Latitude', 'Longitude', 'Elevation (m)',
       'Curie Depth (Km)','Moho Depth (m)',
       'Strike-slip Fault', 'Reverse or Thrust Fault',
       'Right-lateral Fault', 'Nearest Basement',
       'Normal Fault', 'Active Fault','Magnetic Anomaly (nT)',
       'Vertical Gravity Gradient (E)', 'Free Air Anomaly (mGal)',
       'Bouguer Anomaly (mGal)', 'ID',
       'Apparent Geothermal Gradient (°C/Km)']].copy()

for c in df_final.columns:
    if c not in ['ID']:
        df_final[c] = pd.to_numeric(df_final[c], errors='coerce')

# Coordenadas geográficas originales: solo mapas/trazabilidad.
df_final[GEO_LAT_COL] = df_final["Latitude"].copy()
df_final[GEO_LON_COL] = df_final["Longitude"].copy()

# WGS84 -> UTM zona 18N.
transformer = Transformer.from_crs("EPSG:4326", PROJECTED_CRS, always_xy=True)
x_m, y_m = transformer.transform(
    df_final["Longitude"].to_numpy(dtype=float),
    df_final["Latitude"].to_numpy(dtype=float)
)
df_final[PROJECTED_X_COL] = x_m
df_final[PROJECTED_Y_COL] = y_m

# Latitude/Longitude NO entran al modelo
drop_cols = [
    'ID', TARGET_COL, 'tif_name', 'Gradient Weight',
    'Latitude', 'Longitude', GEO_LAT_COL, GEO_LON_COL
]
drop_cols = [c for c in drop_cols if c in df_final.columns]

NO_SCALE_COLS = ['Volcanic domain', 'Volcanic weight']
NO_SCALE_COLS = [c for c in NO_SCALE_COLS if c in df_final.columns]

tabular_cols = [
    c for c in df_final.columns
    if c not in drop_cols
    and pd.api.types.is_numeric_dtype(df_final[c])
    and not df_final[c].isna().all()
]

scale_cols = [c for c in tabular_cols if c not in NO_SCALE_COLS]

print("Columnas tabulares:", tabular_cols)
print("Columnas escaladas:", scale_cols)
print("Columnas NO escaladas:", NO_SCALE_COLS)
print("Coordenadas para mapas (grados):", GEO_LON_COL, GEO_LAT_COL)
print("Coordenadas proyectadas para el modelo:", PROJECTED_X_COL, PROJECTED_Y_COL)
print("CRS proyectado:", PROJECTED_CRS)

df_final = df_final.dropna(
    subset=tabular_cols + [TARGET_COL, 'ID', GEO_LAT_COL, GEO_LON_COL]
).reset_index(drop=True)

print("Registros tras eliminar NaNs:", len(df_final))

# SPLIT train / calibration / validation / test

# Primero se conserva la división original 80/10/10 para validation/test.
train_pool_df, temp_df = train_test_split(
    df_final, test_size=0.20, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42
)

# exclusivamente para calibración conformal.
train_df, cal_df = train_test_split(
    train_pool_df,
    test_size=0.125,
    random_state=42
)

train_df = train_df.reset_index(drop=True).copy()
cal_df   = cal_df.reset_index(drop=True).copy()
val_df   = val_df.reset_index(drop=True).copy()
test_df  = test_df.reset_index(drop=True).copy()

print(
    f"Split -> Train: {len(train_df)} | Calibration: {len(cal_df)} | "
    f"Val: {len(val_df)} | Test: {len(test_df)}"
)

# Copias crudas de los splits.
# Optuna usa train/validation; calibration y test permanecen fuera del objective.
train_raw_split_df = train_df.copy()
cal_raw_split_df = cal_df.copy()
val_raw_split_df = val_df.copy()
test_raw_split_df = test_df.copy()

train_df = add_gradient_weights_lds(
    train_df,
    target_col=TARGET_COL,
    weight_col="Gradient Weight",
    sigma=LDS_SIGMA,
    n_bins=15
)

count_gt_1 = int(np.sum(train_df["Gradient Weight"].values > 1))
percent_gt_1 = 100 * count_gt_1 / len(train_df)
print(f"[LDS train] Valores > 1: {count_gt_1} de {len(train_df)} ({percent_gt_1:.1f}%)")
print("¿Gradient Weight en calibration?:", "Gradient Weight" in cal_df.columns)
print("¿Gradient Weight en val?:", "Gradient Weight" in val_df.columns)
print("¿Gradient Weight en test?:", "Gradient Weight" in test_df.columns)

# Escalado tabular

scaler_tab = StandardScaler()
train_df[scale_cols] = scaler_tab.fit_transform(train_df[scale_cols])
cal_df[scale_cols]   = scaler_tab.transform(cal_df[scale_cols])
val_df[scale_cols]   = scaler_tab.transform(val_df[scale_cols])
test_df[scale_cols]  = scaler_tab.transform(test_df[scale_cols])

joblib.dump(scaler_tab, OUT_DIR / "tabular_scaler.pkl")
with open(OUT_DIR / "tabular_columns.json", "w", encoding="utf-8") as fh:
    json.dump(
        {
            "tabular_cols": tabular_cols,
            "scale_cols": scale_cols,
            "geo_lat_col": GEO_LAT_COL,
            "geo_lon_col": GEO_LON_COL,
            "projected_x_col": PROJECTED_X_COL,
            "projected_y_col": PROJECTED_Y_COL,
            "projected_crs": PROJECTED_CRS,
            "projected_units": "m"
        },
        fh,
        ensure_ascii=False,
        indent=2
    )

# Guardar copia del train original para métricas e imagen stats
train_original_df = train_df.copy()

# WSMOTER tabular
def wsmoter_oversample_regression(
    df,
    feature_cols,
    target_col,
    id_col="ID",
    weight_col="Gradient Weight",
    multiplier=1.35,
    k_neighbors=5,
    weight_power=1.5,
    random_state=42,
    id_mode="anchor"
):
    """
    Implementación práctica tipo WSMOTER para regresión.

    Qué hace:
    - Genera nuevas filas tabulares sintéticas en train.
    - Selecciona muestras ancla con probabilidad proporcional a sus pesos LDS.
    - Interpola las variables tabulares indicadas en feature_cols y el target.
    - Mantiene el vínculo imagen-tabla asignando a cada fila sintética un ID real.

    Nota importante para este proyecto:
    - x_m e y_m entran al modelo, pero NO se interpolan en WSMOTER.
      La fila sintética conserva las coordenadas normalizadas y coordenadas raw
      del ID fuente para mantener coherencia con la imagen Landsat asociada.
    - Val/test no tienen Gradient Weight ni pasan por WSMOTER.
    """
    if multiplier <= 1.0:
        out = df.copy()
        out["is_synthetic"] = 0
        out["source_ID"] = out[id_col].astype(int)
        return out.reset_index(drop=True)

    rng = np.random.default_rng(random_state)

    base = df.reset_index(drop=True).copy()
    n = len(base)
    n_new = int(round((multiplier - 1.0) * n))

    if n_new <= 0 or n < 2:
        base["is_synthetic"] = 0
        base["source_ID"] = base[id_col].astype(int)
        return base.reset_index(drop=True)

    k = min(k_neighbors + 1, n)

    X = base[feature_cols].to_numpy(dtype=np.float32)
    y = base[target_col].to_numpy(dtype=np.float32)

    if weight_col in base.columns:
        w = base[weight_col].to_numpy(dtype=np.float64)
        w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        w = np.maximum(w, 1e-8)
    else:
        # Si no existe weight_col, se pondera por rareza simple.
        y_center = np.median(y)
        mad = np.median(np.abs(y - y_center)) + 1e-8
        w = 1.0 + np.abs(y - y_center) / mad

    probs = np.power(w, weight_power)
    probs = probs / probs.sum()

    nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    nn.fit(X)
    neigh_idx = nn.kneighbors(X, return_distance=False)

    synthetic_rows = []

    for _ in range(n_new):
        i = int(rng.choice(np.arange(n), p=probs))

        candidates = neigh_idx[i]
        candidates = candidates[candidates != i]

        if candidates.size == 0:
            j = i
        else:
            cand_w = w[candidates]
            cand_p = cand_w / cand_w.sum()
            j = int(rng.choice(candidates, p=cand_p))

        lam = rng.uniform(0.0, 1.0)

        xi = X[i]
        xj = X[j]

        yi = y[i]
        yj = y[j]

        x_new = xi + lam * (xj - xi)
        y_new = yi + lam * (yj - yi)

        # Se parte de la fila ancla.
        row = base.iloc[i].copy()

        for col, val in zip(feature_cols, x_new):
            row[col] = float(val)

        row[target_col] = float(y_new)

        if weight_col in base.columns:
            row[weight_col] = float(w[i] + lam * (w[j] - w[i]))

        if id_mode == "neighbor":
            source_id = int(base.iloc[j][id_col])
            # Si se usa neighbor, también se conservan las coordenadas del vecino.
            row[id_col] = source_id
            row["Latitude"] = base.iloc[j]["Latitude"]
            row["Longitude"] = base.iloc[j]["Longitude"]
            row[GEO_LAT_COL] = base.iloc[j][GEO_LAT_COL]
            row[GEO_LON_COL] = base.iloc[j][GEO_LON_COL]
            row[PROJECTED_X_COL] = base.iloc[j][PROJECTED_X_COL]
            row[PROJECTED_Y_COL] = base.iloc[j][PROJECTED_Y_COL]
        else:
            source_id = int(base.iloc[i][id_col])
            row[id_col] = source_id

        row["source_ID"] = source_id
        row["is_synthetic"] = 1

        synthetic_rows.append(row)

    synth_df = pd.DataFrame(synthetic_rows)

    base["is_synthetic"] = 0
    base["source_ID"] = base[id_col].astype(int)

    out = pd.concat([base, synth_df], axis=0, ignore_index=True)
    out = out.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    return out

# Columnas usadas para generar muestras sintéticas
WSMOTER_FEATURE_COLS = [
    c for c in tabular_cols
    if c not in [PROJECTED_X_COL, PROJECTED_Y_COL]
]

print("Columnas usadas por WSMOTER:", WSMOTER_FEATURE_COLS)
print("Columnas tabulares de entrada al modelo:", tabular_cols)

if USE_WSMOTER_TABULAR:
    train_df = wsmoter_oversample_regression(
        df=train_df,
        feature_cols=WSMOTER_FEATURE_COLS,
        target_col=TARGET_COL,
        id_col="ID",
        weight_col="Gradient Weight",
        multiplier=WSMOTER_MULTIPLIER,
        k_neighbors=WSMOTER_K_NEIGHBORS,
        weight_power=WSMOTER_WEIGHT_POWER,
        random_state=SEED,
        id_mode=WSMOTER_ID_MODE
    )
else:
    train_df["is_synthetic"] = 0
    train_df["source_ID"] = train_df["ID"].astype(int)

cal_df["is_synthetic"] = 0
cal_df["source_ID"] = cal_df["ID"].astype(int)

val_df["is_synthetic"] = 0
val_df["source_ID"] = val_df["ID"].astype(int)

test_df["is_synthetic"] = 0
test_df["source_ID"] = test_df["ID"].astype(int)

train_original_df["is_synthetic"] = 0
train_original_df["source_ID"] = train_original_df["ID"].astype(int)

# Seguridad: calibration, validación y test no deben tener pesos LDS.
cal_df = cal_df.drop(columns=["Gradient Weight"], errors="ignore")
val_df = val_df.drop(columns=["Gradient Weight"], errors="ignore")
test_df = test_df.drop(columns=["Gradient Weight"], errors="ignore")

print(
    f"Train después de WSMOTER: {len(train_df)} "
    f"(originales={int((train_df['is_synthetic'] == 0).sum())}, "
    f"sintéticas={int((train_df['is_synthetic'] == 1).sum())})"
)
print(f"Calibration: {len(cal_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("¿Gradient Weight en train?:", "Gradient Weight" in train_df.columns)
print("¿Gradient Weight en val?:", "Gradient Weight" in val_df.columns)
print("¿Gradient Weight en test?:", "Gradient Weight" in test_df.columns)

# Transformación del target
for _df in [train_df, train_original_df, cal_df, val_df, test_df]:
    _df[TARGET_COL + "_raw"] = _df[TARGET_COL].copy()

min_train_target = train_original_df[TARGET_COL].min()
target_shift = 0.0

if min_train_target <= -1:
    target_shift = float(abs(min_train_target) + 1.0)
    print(f"Aplicando shift al target: {target_shift:.6f}")

    for _df in [train_df, train_original_df, cal_df, val_df, test_df]:
        _df[TARGET_COL] += target_shift

for _df in [train_df, train_original_df, cal_df, val_df, test_df]:
    _df[TARGET_COL] = np.log1p(_df[TARGET_COL].astype(float))

scaler_y = StandardScaler()
train_df[[TARGET_COL]] = scaler_y.fit_transform(train_df[[TARGET_COL]])

# Mismo escalador para train original, val y test
train_original_df[[TARGET_COL]] = scaler_y.transform(train_original_df[[TARGET_COL]])
cal_df[[TARGET_COL]] = scaler_y.transform(cal_df[[TARGET_COL]])
val_df[[TARGET_COL]] = scaler_y.transform(val_df[[TARGET_COL]])
test_df[[TARGET_COL]] = scaler_y.transform(test_df[[TARGET_COL]])

joblib.dump(scaler_y, OUT_DIR / "target_scaler.pkl")
joblib.dump({"shift": target_shift}, OUT_DIR / "target_shift.json")

# Caché de imágenes
build_image_cache(
    df_final,
    desc=f"Cacheando imágenes {TARGET_SIZE[0]}x{TARGET_SIZE[1]}"
)

# Estadísticas de imagen
image_mean, image_std = compute_image_stats_train(train_original_df)
thermal_mean, thermal_std = compute_thermal_stats_train(train_original_df)

joblib.dump(
    {"mean": image_mean, "std": image_std, "band_order": BAND_ORDER},
    OUT_DIR / "image_stats.pkl"
)

thermal_channel_names = [BAND_ORDER[i] for i in THERMAL_DERIVED_IDXS]

joblib.dump(
    {
        "mean": thermal_mean,
        "std": thermal_std,
        "channels": thermal_channel_names
    },
    OUT_DIR / "thermal_stats.pkl"
)

# Guardar trazabilidad de los splits
train_df.to_csv(OUT_DIR / "train_wsmoter.csv", index=False)
train_original_df.to_csv(OUT_DIR / "train_original_no_wsmoter.csv", index=False)
cal_df.to_csv(OUT_DIR / "calibration.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print("Image mean:", image_mean)
print("Image std:", image_std)
print("Thermal/derived channels:", thermal_channel_names)
print("Thermal/derived mean:", thermal_mean)
print("Thermal/derived std:", thermal_std)


In [ ]:
import shutil

tiff_dir = "/kaggle/working/LANDSAT_GEOTIFF"

shutil.rmtree(tiff_dir)
print("TIFF eliminados.")

In [ ]:
# Cell X: diagrama de pesos LDS por bins del gradiente geotérmico

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# DataFrame a usar para el gráfico
plot_df = train_original_df.copy()

target_col = TARGET_COL
weight_col = "Gradient Weight"

# Si definiste un número fijo de bins, úsalo.
# Si no existe, se usan 15 bins (16 bordes)
n_bins_plot = LDS_N_BINS if "LDS_N_BINS" in globals() else 10

# Verificación básica
if weight_col not in plot_df.columns:
    raise ValueError(
        f"No existe la columna '{weight_col}' en plot_df. "
        "Asegúrate de haber ejecutado add_gradient_weights_lds(...) antes de esta celda."
    )

# Crear bordes de bins
bin_edges = np.linspace(
    plot_df[target_col].min(),
    plot_df[target_col].max(),
    num=n_bins_plot + 1
)

# Asignar cada valor a su bin correspondiente
plot_df["bin"] = np.digitize(
    plot_df[target_col],
    bins=bin_edges,
    right=True
)

# Corregir extremos para que siempre queden dentro de [1, n_bins_plot]
plot_df["bin"] = plot_df["bin"].clip(1, n_bins_plot)

# Peso promedio por bin
bin_to_weight = plot_df.groupby("bin")[weight_col].mean()

# Asegurar que todos los bins existan, aunque estén vacíos
all_bins = range(1, n_bins_plot + 1)
bin_to_weight_full = bin_to_weight.reindex(all_bins).fillna(0.0)

# Normalización de pesos a [0, 1]
w_min = bin_to_weight_full.min()
w_max = bin_to_weight_full.max()

if w_max > w_min:
    bin_weights_norm = (bin_to_weight_full - w_min) / (w_max - w_min)
else:
    bin_weights_norm = pd.Series(0.5, index=bin_to_weight_full.index)

# Figura
fig, ax = plt.subplots(figsize=(10, 6))

# Dibujar histograma
n, bins, patches = ax.hist(
    plot_df[target_col],
    bins=bin_edges,
    edgecolor="black"
)

# Colormap
cmap = plt.cm.jet
norm = mcolors.Normalize(
    vmin=bin_weights_norm.min(),
    vmax=bin_weights_norm.max()
)

# Colorear cada bin según su peso medio normalizado
for i, patch in enumerate(patches):
    bin_number = i + 1
    color = cmap(norm(bin_weights_norm.loc[bin_number]))
    patch.set_facecolor(color)

# Etiquetas
ax.set_xlabel("Gradiente Geotérmico Aparente (°C/Km)")
ax.set_ylabel("Frecuencia")
ax.set_title("Histograma del Gradiente Geotérmico Aparente coloreado por peso LDS")

# Barra de color
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(bin_weights_norm.values)

cbar = fig.colorbar(
    sm,
    ax=ax,
    ticks=np.linspace(bin_weights_norm.min(), bin_weights_norm.max(), num=5)
)
cbar.ax.set_yticklabels(["Peso bajo", "", "", "", "Peso alto"])
cbar.set_label("Peso promedio normalizado por bin")

plt.tight_layout()
plt.show()

# Opcional: eliminar columna temporal
plot_df.drop(columns=["bin"], inplace=True, errors="ignore")

In [ ]:
# Cell X: histograma del target original coloreado por peso LDS real

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plot_df = train_original_df.copy()

weight_col = "Gradient Weight"
target_raw_col = TARGET_COL + "_raw"

if weight_col not in plot_df.columns:
    raise ValueError(
        f"No existe '{weight_col}' en train_original_df. "
        "Ejecuta primero add_gradient_weights_lds(...)."
    )

if target_raw_col not in plot_df.columns:
    raise ValueError(
        f"No existe '{target_raw_col}'. "
        "Esta columna debe crearse antes de transformar el target."
    )

n_bins_plot = LDS_N_BINS if "LDS_N_BINS" in globals() else 15

# Bordes sobre el target original, no sobre el target escalado
bin_edges = np.linspace(
    plot_df[target_raw_col].min(),
    plot_df[target_raw_col].max(),
    num=n_bins_plot + 1
)

plot_df["bin"] = np.digitize(
    plot_df[target_raw_col],
    bins=bin_edges,
    right=True
)

plot_df["bin"] = plot_df["bin"].clip(1, n_bins_plot)

# Peso LDS promedio real por bin
bin_to_weight = plot_df.groupby("bin")[weight_col].mean()
all_bins = range(1, n_bins_plot + 1)
bin_to_weight_full = bin_to_weight.reindex(all_bins)

valid_weights = bin_to_weight_full.dropna()

if len(valid_weights) == 0:
    raise ValueError("No hay pesos válidos para graficar.")

w_min = valid_weights.min()
w_max = valid_weights.max()

if w_max == w_min:
    w_min -= 1e-6
    w_max += 1e-6

fig, ax = plt.subplots(figsize=(10, 6))

counts, bins, patches = ax.hist(
    plot_df[target_raw_col],
    bins=bin_edges,
    edgecolor="black"
)

cmap = plt.cm.jet
norm = mcolors.Normalize(vmin=w_min, vmax=w_max)

for i, patch in enumerate(patches):
    bin_number = i + 1
    weight_value = bin_to_weight_full.loc[bin_number]

    if pd.isna(weight_value):
        patch.set_facecolor("lightgray")
    else:
        patch.set_facecolor(cmap(norm(weight_value)))

ax.set_xlabel("Gradiente Geotérmico Aparente original (°C/km)")
ax.set_ylabel("Frecuencia")
ax.set_title("Histograma del Gradiente Geotérmico Aparente coloreado por peso LDS real")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(valid_weights.values)

cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Peso LDS promedio real por bin")

plt.tight_layout()
plt.show()

In [ ]:
# Cell X: diagnóstico numérico de pesos LDS por bin

diagnostic_df = plot_df.groupby("bin").agg(
    y_min=(target_raw_col, "min"),
    y_max=(target_raw_col, "max"),
    frecuencia=(target_raw_col, "count"),
    peso_promedio=(weight_col, "mean"),
    peso_min=(weight_col, "min"),
    peso_max=(weight_col, "max")
).reset_index()

display(diagnostic_df)

In [ ]:
# Cell 12
# El backbone visual es OlmoEarth v1.1 Tiny, cargado desde Hugging Face.

print("Backbone visual:", OLMOEARTH_MODEL_ID)
print("Pretrained:", PRETRAINED_OLMOEARTH)
print("Patch size OlmoEarth:", OLMOEARTH_PATCH_SIZE)
print("Input resolution OlmoEarth:", OLMOEARTH_INPUT_RES)
print("Tamaño interno OlmoEarth:", OLMOEARTH_TARGET_SIZE)
print("Canales GeoTIFF originales:", N_IMAGE_CHANNELS)
print("Canales rama térmica/derivados:", N_THERMAL_DERIVED_CHANNELS)
print("Bandas GeoTIFF:", BAND_ORDER)
print("Orden Landsat esperado por OlmoEarth:", OLMOEARTH_LANDSAT_ORDER)
print("Rama térmica/derivados:", [BAND_ORDER[i] for i in THERMAL_DERIVED_IDXS])

In [ ]:
class MultibandPairAugment:
    def __init__(
        self,
        p_hflip=0.5,
        p_vflip=0.5,
        p_rot90=0.5,
        p_noise=0.35,
        noise_sigma_image=0.025,
        noise_sigma_thermal=0.020,
        clip_value=5.0
    ):
        self.p_hflip = p_hflip
        self.p_vflip = p_vflip
        self.p_rot90 = p_rot90
        self.p_noise = p_noise
        self.noise_sigma_image = noise_sigma_image
        self.noise_sigma_thermal = noise_sigma_thermal
        self.clip_value = clip_value

    def __call__(self, optical, thermal):
        """
        optical: tensor [C, H, W]
        thermal: tensor [C_aux, H, W]
        """
        # Flip horizontal
        if torch.rand(1).item() < self.p_hflip:
            optical = torch.flip(optical, dims=[2])
            thermal = torch.flip(thermal, dims=[2])

        # Flip vertical
        if torch.rand(1).item() < self.p_vflip:
            optical = torch.flip(optical, dims=[1])
            thermal = torch.flip(thermal, dims=[1])

        # Rotación en múltiplos de 90 grados
        if torch.rand(1).item() < self.p_rot90:
            k = torch.randint(0, 4, (1,)).item()
            optical = torch.rot90(optical, k=k, dims=[1, 2])
            thermal = torch.rot90(thermal, k=k, dims=[1, 2])

        # Ruido gaussiano suave, después de normalizar
        if torch.rand(1).item() < self.p_noise:
            optical = optical + torch.randn_like(optical) * self.noise_sigma_image
            thermal = thermal + torch.randn_like(thermal) * self.noise_sigma_thermal

            optical = torch.clamp(optical, -self.clip_value, self.clip_value)
            thermal = torch.clamp(thermal, -self.clip_value, self.clip_value)

        return optical, thermal


class MultimodalDataset(Dataset):
    def __init__(self, df, train=False, augment=None):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.augment = augment

        image_stats = joblib.load(OUT_DIR / "image_stats.pkl")
        self.img_mean = np.asarray(image_stats["mean"], dtype=np.float32)
        self.img_std  = np.asarray(image_stats["std"], dtype=np.float32)

        thermal_stats = joblib.load(OUT_DIR / "thermal_stats.pkl")
        self.th_mean = np.asarray(thermal_stats["mean"], dtype=np.float32)
        self.th_std  = np.asarray(thermal_stats["std"], dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # En filas sintéticas WSMOTER, ID corresponde a una imagen real
        cached_image = get_cached_full_image(int(row.ID))

        # optical = stack completo de 9/10 canales
        # thermal = Band61 + NDVI + Band3/Band1 + Band5/Band7
        optical = cached_image.copy()
        thermal = cached_image[THERMAL_DERIVED_IDXS].copy()

        # Normalización por canal para la rama visual
        optical = optical.astype(np.float32)

        for b in range(optical.shape[0]):
            optical[b] = (optical[b] - self.img_mean[b]) / (self.img_std[b] + 1e-8)

        optical = np.clip(optical, -5.0, 5.0)

        # Normalización por canal para rama térmica/derivados
        thermal = thermal.astype(np.float32)

        for b in range(thermal.shape[0]):
            thermal[b] = (thermal[b] - self.th_mean[b]) / (self.th_std[b] + 1e-8)

        thermal = np.clip(thermal, -5.0, 5.0)

        optical = torch.from_numpy(optical.astype(np.float32))   # [C, H, W]
        thermal = torch.from_numpy(thermal.astype(np.float32))   # [4, H, W]

        # Las aumentaciones espaciales y el ruido suave se aplican SOLO
        # a las muestras sintéticas generadas por WSMOTER.
        is_synthetic = int(row["is_synthetic"]) if "is_synthetic" in row.index else 0

        if self.train and self.augment is not None and is_synthetic == 1:
            optical, thermal = self.augment(optical, thermal)

        tab_vals = row[tabular_cols].to_numpy(dtype=np.float32, copy=True)
        tab = torch.from_numpy(tab_vals)

        y = torch.tensor(np.float32(row[TARGET_COL]), dtype=torch.float32)

        return {
            "optical": optical,
            "thermal": thermal,
            "tabular": tab,
            "target": y,
            "id": int(row.ID),
            "source_id": int(row.source_ID) if "source_ID" in row.index else int(row.ID),
            "is_synthetic": is_synthetic
        }


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from olmoearth_pretrain_minimal import ModelID, load_model_from_id
    from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.datatypes import MaskedOlmoEarthSample
except Exception as e:
    raise ImportError(
        "No se pudo importar olmoearth-pretrain-minimal. "
        "Ejecuta la celda de instalación o instala: pip install olmoearth-pretrain-minimal"
    ) from e


def _valid_num_groups(channels, preferred=(16, 8, 4, 2, 1)):
    for g in preferred:
        if g <= channels and channels % g == 0:
            return g
    return 1

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2)

    def forward(self, x, return_attention=False):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        out = x * attn

        if return_attention:
            return out, attn

        return out

class SEVector(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(1, channels // reduction)

        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        s = self.fc(x)
        return x * s

class SqueezeExcite2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(8, channels // reduction)

        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.GELU(),
            nn.Linear(mid, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1).flatten(1)
        s = self.fc(s).unsqueeze(-1).unsqueeze(-1)
        return x * s

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size,
                stride=stride,
                padding=padding,
                bias=False
            ),
            nn.GroupNorm(_valid_num_groups(out_ch), out_ch),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)

class ThermalResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop=0.0):
        super().__init__()

        self.conv1 = ConvGNAct(
            in_ch,
            out_ch,
            kernel_size=3,
            stride=stride,
            padding=1
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.GroupNorm(_valid_num_groups(out_ch), out_ch)
        )

        self.se = SqueezeExcite2D(out_ch, reduction=8)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()

        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.GroupNorm(_valid_num_groups(out_ch), out_ch)
            )
        else:
            self.skip = nn.Identity()

        self.act = nn.GELU()

    def forward(self, x):
        identity = self.skip(x)

        out = self.conv1(x)
        out = self.conv2(out)
        out = self.se(out)
        out = self.drop(out)

        out = out + identity
        return self.act(out)

class MultiScaleTokenize(nn.Module):
    def __init__(self, in_channels_list, d_model=512, token_pool_size=(4, 4)):
        super().__init__()

        assert len(in_channels_list) > 0

        self.token_pool_size = token_pool_size
        self.projs = nn.ModuleList([
            nn.Conv2d(in_ch, d_model, kernel_size=1)
            for in_ch in in_channels_list
        ])

    def forward(self, feats):
        tokens_list = []

        for f, proj in zip(feats, self.projs):
            x = proj(f)
            x = F.adaptive_avg_pool2d(x, self.token_pool_size)

            B, C, H, W = x.shape

            t = x.view(B, C, H * W).permute(0, 2, 1)
            tokens_list.append(t)

        tokens = torch.cat(tokens_list, dim=1)
        return tokens


In [ ]:
class OlmoEarthLandsat7Backbone(nn.Module):
    """
    Adaptador mínimo para usar OlmoEarth v1.1 Tiny con los GeoTIFF Landsat 7 del notebook.

    Entrada del notebook:  [B, N_IMAGE_CHANNELS, H, W]
    Entrada OlmoEarth:     [B, H, W, T=1, 11] en orden Landsat:
                           [B8, B1, B2, B3, B4, B5, B6, B7, B9, B10, B11]

    Importante: tus imágenes NO son multitemporales. Se usa T=1 solo porque la
    interfaz de OlmoEarth representa las imágenes como una secuencia temporal.
    No se repite la imagen ni se simulan varias fechas.

    Para Landsat 7 se hace mapeo por equivalencia espectral hacia la convención
    Landsat esperada por OlmoEarth, no por número literal de banda.

    Las bandas no disponibles en Landsat 7 o en el GeoTIFF se rellenan con cero.
    Los índices derivados, como NDVI y cocientes, no se insertan en OlmoEarth;
    permanecen en la rama térmica/derivados ya existente.
    """
    _MODEL_IDS = {
        "nano": "OLMOEARTH_V1_1_NANO",
        "tiny": "OLMOEARTH_V1_1_TINY",
        "base": "OLMOEARTH_V1_1_BASE",
    }

    def __init__(
        self,
        model_size="tiny",
        pretrained=True,
        source_band_order=None,
        landsat_band_order=None,
        target_size=(224, 224),
        patch_size=8,
        input_res=30,
        embed_dim=192,
        use_thermal_in_backbone=True,
    ):
        super().__init__()

        if model_size not in self._MODEL_IDS:
            raise ValueError(f"model_size={model_size!r} no soportado. Usa: {list(self._MODEL_IDS)}")

        self.model_size = model_size
        self.pretrained = pretrained
        self.source_band_order = list(source_band_order if source_band_order is not None else BAND_ORDER)
        self.landsat_band_order = list(landsat_band_order if landsat_band_order is not None else OLMOEARTH_LANDSAT_ORDER)
        self.target_size = tuple(target_size) if target_size is not None else None
        self.patch_size = int(patch_size)
        self.input_res = int(input_res)
        self.embed_dim = int(embed_dim)
        self.use_thermal_in_backbone = bool(use_thermal_in_backbone)

        model_id = getattr(ModelID, self._MODEL_IDS[model_size])
        full_model = load_model_from_id(model_id, load_weights=pretrained)
        self.encoder = full_model.encoder

        self._src_index = {name: i for i, name in enumerate(self.source_band_order)}
        self._dst_index = {name: i for i, name in enumerate(self.landsat_band_order)}

        # Mapeo Landsat 7 ETM+ -> convención Landsat usada por OlmoEarth.
        # Se mapea por equivalencia espectral:
        #   ETM+ B1 blue  -> Olmo/Landsat B2 blue
        #   ETM+ B2 green -> Olmo/Landsat B3 green
        #   ETM+ B3 red   -> Olmo/Landsat B4 red
        #   ETM+ B4 NIR   -> Olmo/Landsat B5 NIR
        #   ETM+ B5 SWIR1 -> Olmo/Landsat B6 SWIR1
        #   ETM+ B7 SWIR2 -> Olmo/Landsat B7 SWIR2
        self._band_mapping = {
            "Band1_SRF_REF": "B2",
            "Band2_SRF_REF": "B3",
            "Band3_SRF_REF": "B4",
            "Band4_SRF_REF": "B5",
            "Band5_SRF_REF": "B6",
            "Band7_SRF_REF": "B7",
        }

        # Por defecto se deja la banda térmica fuera del backbone preentrenado
        # porque Band61_TOA_BT tiene una distribución física distinta a las reflectancias.
        # Sigue entrando por la rama térmica/derivados del modelo. Si se activa, se ubica
        # como banda térmica B10 y B11 queda en cero.
        if self.use_thermal_in_backbone:
            self._band_mapping["Band61_TOA_BT"] = "B10"

        missing = [src for src in self._band_mapping if src not in self._src_index]
        if missing:
            print(
                "Advertencia OlmoEarth: no se encontraron estas bandas en el GeoTIFF "
                f"y se rellenarán con cero: {missing}"
            )

    def _to_olmo_landsat_tensor(self, img):
        """
        img: [B, N_IMAGE_CHANNELS, H, W]
        return: [B, 11, H, W] en el orden Landsat esperado por OlmoEarth.
        """
        B, _, H, W = img.shape
        x = img.new_zeros((B, len(self.landsat_band_order), H, W))

        for src_name, dst_name in self._band_mapping.items():
            if src_name in self._src_index and dst_name in self._dst_index:
                x[:, self._dst_index[dst_name], :, :] = img[:, self._src_index[src_name], :, :]

        return x

    @staticmethod
    def _extract_modality_tokens(out, modality_name="landsat"):
        if not isinstance(out, dict) or "tokens_and_masks" not in out:
            raise RuntimeError(
                "La salida del encoder OlmoEarth no tiene la clave 'tokens_and_masks'. "
                "Revisa la versión de olmoearth-pretrain-minimal."
            )

        tam = out["tokens_and_masks"]

        if isinstance(tam, dict):
            tokens = tam.get(modality_name, None)
        else:
            tokens = getattr(tam, modality_name, None)

        if tokens is None:
            available = []
            if isinstance(tam, dict):
                available = list(tam.keys())
            else:
                available = [k for k in dir(tam) if not k.startswith("_")]

            raise AttributeError(
                f"No se encontraron tokens para la modalidad {modality_name!r}. "
                f"Campos disponibles: {available}"
            )

        return tokens

    def forward(self, img):
        x = self._to_olmo_landsat_tensor(img)

        if self.target_size is not None and x.shape[-2:] != self.target_size:
            x = F.interpolate(
                x,
                size=self.target_size,
                mode="bilinear",
                align_corners=False
            )

        B, C, H, W = x.shape

        # OlmoEarth usa el formato [B, H, W, T, C].
        # Tus imágenes son de una sola fecha; por eso T=1.
        # No se duplica la imagen ni se crea una secuencia temporal artificial.
        x_bhwtc = x.permute(0, 2, 3, 1).unsqueeze(3).contiguous()

        # Timestamp constante/ficticio: no aporta temporalidad; solo cumple la interfaz.
        timestamps = torch.zeros(B, 1, 3, dtype=torch.long, device=x.device)

        # v1.1 usa un solo bandset por modalidad: máscara [B, H, W, T=1, 1].
        mask = torch.zeros(B, H, W, 1, 1, dtype=torch.bool, device=x.device)

        sample = MaskedOlmoEarthSample(
            timestamps=timestamps,
            landsat=x_bhwtc,
            landsat_mask=mask,
        )

        out = self.encoder(
            sample,
            patch_size=self.patch_size,
            input_res=self.input_res,
            fast_pass=True
        )

        tokens = self._extract_modality_tokens(out, modality_name="landsat")

        # Conservar la estructura espacial de los embeddings OlmoEarth
        spatial_tokens = tokens

        while spatial_tokens.ndim > 4:
            spatial_tokens = spatial_tokens.mean(dim=-2)

        if spatial_tokens.ndim == 4:
            # Formato esperado: [B, pH, pW, D]
            pass

        elif spatial_tokens.ndim == 3:
            # Compatibilidad con versiones que devuelvan [B, N, D].
            n_tokens = spatial_tokens.shape[1]
            side = int(round(n_tokens ** 0.5))

            if side * side != n_tokens:
                raise RuntimeError(
                    "No se pudo reconstruir el mapa espacial de OlmoEarth: "
                    f"se recibieron {n_tokens} tokens, que no forman una grilla cuadrada."
                )

            spatial_tokens = spatial_tokens.reshape(
                B,
                side,
                side,
                spatial_tokens.shape[-1]
            )

        else:
            raise RuntimeError(
                "Forma inesperada de tokens OlmoEarth para atención espacial: "
                f"{tuple(spatial_tokens.shape)}"
            )

        seq_tokens = spatial_tokens.reshape(
            B,
            -1,
            spatial_tokens.shape[-1]
        )

        z = seq_tokens.mean(dim=1)

        # Se devuelve además el mapa espacial [B, pH, pW, D].
        return z, seq_tokens, spatial_tokens


class OlmoEarthTiny_Multimodal_BiAttn_Residual(nn.Module):
    def __init__(
        self,
        n_tabular,
        n_image_channels=9,
        n_thermal_channels=4,
        pretrained=True,
        d_model=512,
        n_heads=8,
        dropout=0.25,
        token_pool_size=(4, 4),
        img_token_layers=2,
        thermal_token_layers=1,
        img_dim_feedforward=2048,
        thermal_dim_feedforward=None,
        tab_hidden=256,
        head_hidden_1=768,
        head_hidden_2=256,
        se_fusion_reduction=8,
        spatial_attention_kernel=7
    ):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} debe ser divisible por n_heads={n_heads}")

        if thermal_dim_feedforward is None:
            thermal_dim_feedforward = 4 * d_model

        # Rama visual principal: OlmoEarth v1.1 Tiny.
        self.backbone = OlmoEarthLandsat7Backbone(
            model_size=OLMOEARTH_MODEL_SIZE,
            pretrained=pretrained,
            source_band_order=BAND_ORDER,
            landsat_band_order=OLMOEARTH_LANDSAT_ORDER,
            target_size=OLMOEARTH_TARGET_SIZE,
            patch_size=OLMOEARTH_PATCH_SIZE,
            input_res=OLMOEARTH_INPUT_RES,
            embed_dim=OLMOEARTH_EMBED_DIM,
            use_thermal_in_backbone=OLMOEARTH_USE_THERMAL_IN_BACKBONE,
        )

        olmo_dim = OLMOEARTH_EMBED_DIM

        # Atención espacial sobre el mapa de embeddings de OlmoEarth
        self.spatial_attn = SpatialAttention(
            kernel_size=spatial_attention_kernel
        )

        self.img_token_proj = nn.Sequential(
            nn.Linear(olmo_dim, d_model),
            nn.GELU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=img_dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.img_token_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=img_token_layers
        )

        self.img_vec_proj = nn.Sequential(
            nn.Linear(olmo_dim, d_model),
            nn.GELU()
        )

        self.img_token_ln = nn.LayerNorm(d_model)
        self.img_vec_ln = nn.LayerNorm(d_model)

        # Rama tabular
        self.tab_mlp = nn.Sequential(
            nn.Linear(n_tabular, tab_hidden),
            nn.LayerNorm(tab_hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(tab_hidden, tab_hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(tab_hidden, d_model),
            nn.GELU()
        )

        self.tab_ln = nn.LayerNorm(d_model)

        # Rama térmica + derivados
        self.thermal_stem = ConvGNAct(
            n_thermal_channels,
            32,
            kernel_size=3,
            stride=2,
            padding=1
        )

        self.thermal_stage1 = ThermalResBlock(32, 64, stride=2, drop=0.05)
        self.thermal_stage2 = ThermalResBlock(64, 128, stride=2, drop=0.05)
        self.thermal_stage3 = ThermalResBlock(128, 256, stride=2, drop=0.10)

        self.thermal_context = nn.Sequential(
            ThermalResBlock(256, 256, stride=1, drop=0.10),
            ThermalResBlock(256, 256, stride=1, drop=0.10)
        )

        self.thermal_token_projs = nn.ModuleList([
            nn.Sequential(nn.Linear(32 * 2, d_model), nn.GELU()),
            nn.Sequential(nn.Linear(64 * 2, d_model), nn.GELU()),
            nn.Sequential(nn.Linear(128 * 2, d_model), nn.GELU()),
            nn.Sequential(nn.Linear(256 * 2, d_model), nn.GELU()),
        ])

        thermal_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=thermal_dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.thermal_token_encoder = nn.TransformerEncoder(
            thermal_encoder_layer,
            num_layers=thermal_token_layers
        )

        self.thermal_ln = nn.LayerNorm(d_model)

        # Atención cruzada imagen <-> tabla
        self.attn_img_to_tab = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True,
            dropout=dropout
        )

        self.attn_tab_to_img = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True,
            dropout=dropout
        )

        # fusión por concatenación + proyección
        self.img_cross_fuse = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model)
        )

        self.tab_cross_fuse = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model)
        )

        self.concat_gate = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model * 3)
        )

        self.se_fusion = SEVector(d_model * 3, reduction=se_fusion_reduction)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Linear(d_model * 3, head_hidden_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(head_hidden_1, head_hidden_2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),

            nn.Linear(head_hidden_2, 1)
        )

        self._init_new_weights()

    def _init_new_weights(self):
        """
        Inicializa solo los módulos nuevos.
        No reinicializa el encoder OlmoEarth preentrenado.
        """
        modules_to_init = [
            self.spatial_attn,
            self.img_token_proj,
            self.img_token_encoder,
            self.img_vec_proj,
            self.img_token_ln,
            self.img_vec_ln,

            self.tab_mlp,
            self.tab_ln,

            self.thermal_stem,
            self.thermal_stage1,
            self.thermal_stage2,
            self.thermal_stage3,
            self.thermal_context,
            self.thermal_token_projs,
            self.thermal_token_encoder,
            self.thermal_ln,

            self.attn_img_to_tab,
            self.attn_tab_to_img,

            self.img_cross_fuse,
            self.tab_cross_fuse,
            self.concat_gate,

            self.se_fusion,
            self.head
        ]

        for module in modules_to_init:
            for m in module.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

                elif isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight,
                        mode="fan_out",
                        nonlinearity="relu"
                    )
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def _image_forward(self, img, return_attention=False):
        """
        img: [B, N_IMAGE_CHANNELS, H, W]

        Cuando return_attention=True también devuelve el mapa espacial
        aprendido por SpatialAttention para análisis/figuras del artículo.
        El comportamiento normal de entrenamiento no cambia.
        """
        _, _, olmo_spatial_tokens = self.backbone(img)

        # OlmoEarth entrega [B, pH, pW, D].
        # SpatialAttention espera [B, C, H, W].
        spatial_map = olmo_spatial_tokens.permute(
            0, 3, 1, 2
        ).contiguous()

        if return_attention:
            spatial_map, spatial_weights = self.spatial_attn(
                spatial_map,
                return_attention=True
            )
        else:
            spatial_map = self.spatial_attn(spatial_map)
            spatial_weights = None

        attended_olmo_tokens = spatial_map.flatten(2).transpose(
            1, 2
        ).contiguous()

        z_olmo_attended = attended_olmo_tokens.mean(dim=1)

        tokens = self.img_token_proj(attended_olmo_tokens)
        tokens = self.img_token_encoder(tokens)
        tokens = self.img_token_ln(tokens)

        z_img = self.img_vec_proj(z_olmo_attended)
        z_img = self.img_vec_ln(z_img)

        if return_attention:
            return z_img, tokens, spatial_weights

        return z_img, tokens

    def _thermal_forward(self, thermal):
        """
        thermal: [B, 4, H, W]
        """
        t1 = self.thermal_stem(thermal)
        t2 = self.thermal_stage1(t1)
        t3 = self.thermal_stage2(t2)
        t4 = self.thermal_stage3(t3)
        t4 = self.thermal_context(t4)

        thermal_feats = [t1, t2, t3, t4]
        thermal_tokens = []

        for f, proj in zip(thermal_feats, self.thermal_token_projs):
            th_avg = F.adaptive_avg_pool2d(f, 1).flatten(1)
            th_max = F.adaptive_max_pool2d(f, 1).flatten(1)

            token = proj(torch.cat([th_avg, th_max], dim=1)).unsqueeze(1)
            thermal_tokens.append(token)

        thermal_tokens = torch.cat(thermal_tokens, dim=1)
        thermal_tokens = self.thermal_token_encoder(thermal_tokens)

        z_th = thermal_tokens.mean(dim=1)
        z_th = self.thermal_ln(z_th)

        return z_th

    def forward(self, img, thermal, tab, return_attention=False):
        # Rama imagen OlmoEarth v1.1 Tiny
        if return_attention:
            z_img, img_tokens, spatial_weights = self._image_forward(
                img,
                return_attention=True
            )
        else:
            z_img, img_tokens = self._image_forward(img)
            spatial_weights = None

        # Rama tabular
        z_tab = self.tab_mlp(tab)
        z_tab = self.tab_ln(z_tab)

        # Atención bidireccional imagen - tabla
        tab_tokens = z_tab.unsqueeze(1)

        if return_attention:
            img_attn_out, img_to_tab_weights = self.attn_img_to_tab(
                query=img_tokens,
                key=tab_tokens,
                value=tab_tokens,
                need_weights=True,
                average_attn_weights=False
            )

            tab_attn_out, tab_to_img_weights = self.attn_tab_to_img(
                query=tab_tokens,
                key=img_tokens,
                value=img_tokens,
                need_weights=True,
                average_attn_weights=False
            )
        else:
            img_attn_out, _ = self.attn_img_to_tab(
                query=img_tokens,
                key=tab_tokens,
                value=tab_tokens,
                need_weights=False
            )

            tab_attn_out, _ = self.attn_tab_to_img(
                query=tab_tokens,
                key=img_tokens,
                value=img_tokens,
                need_weights=False
            )
            img_to_tab_weights = None
            tab_to_img_weights = None

        z_img_att = img_attn_out.mean(dim=1)
        z_tab_att = tab_attn_out.squeeze(1)

        z_img_final = self.img_cross_fuse(
            torch.cat([z_img, z_img_att], dim=1)
        )

        z_tab_final = self.tab_cross_fuse(
            torch.cat([z_tab, z_tab_att], dim=1)
        )

        # Rama térmica + derivados
        z_th = self._thermal_forward(thermal)

        # Gates + SE + regresión
        fusion_pre = torch.cat(
            [z_img_final, z_tab_final, z_th],
            dim=1
        )

        gate_raw = self.concat_gate(fusion_pre)
        g_img_raw, g_tab_raw, g_th_raw = gate_raw.chunk(3, dim=1)

        scale_img = 1.0 + 0.5 * torch.tanh(g_img_raw)
        scale_tab = 1.0 + 0.5 * torch.tanh(g_tab_raw)
        scale_th = 1.0 + 0.5 * torch.tanh(g_th_raw)

        fusion = torch.cat(
            [
                z_img_final * scale_img,
                z_tab_final * scale_tab,
                z_th * scale_th
            ],
            dim=1
        )

        fusion = self.se_fusion(fusion)
        out = self.head(fusion).squeeze(1)

        if return_attention:
            attention_info = {
                "spatial": spatial_weights,
                "img_to_tab": img_to_tab_weights,
                "tab_to_img": tab_to_img_weights,
                "image_token_grid": tuple(spatial_weights.shape[-2:])
            }
            return out, attention_info

        return out


# Alias para mantener compatibilidad con partes del notebook que usaban el nombre anterior.
ConvNeXt9_Multimodal_BiAttn_Residual = OlmoEarthTiny_Multimodal_BiAttn_Residual


In [ ]:
target_scaler = joblib.load(OUT_DIR / "target_scaler.pkl")
target_shift = joblib.load(OUT_DIR / "target_shift.json")["shift"]

train_augment = MultibandPairAugment(
    p_hflip=AUG_P_HFLIP,
    p_vflip=AUG_P_VFLIP,
    p_rot90=AUG_P_ROT90,
    p_noise=AUG_P_NOISE,
    noise_sigma_image=AUG_NOISE_SIGMA_IMAGE,
    noise_sigma_thermal=AUG_NOISE_SIGMA_THERMAL,
    clip_value=AUG_CLIP_VALUE
) if USE_IMAGE_AUGMENTATION else None

train_dataset = MultimodalDataset(
    train_df,
    train=True,
    augment=train_augment
)

# Para métricas de train se usa el train original, sin filas sintéticas WSMOTER.
train_eval_dataset = MultimodalDataset(
    train_original_df,
    train=False,
    augment=None
)

cal_dataset = MultimodalDataset(
    cal_df,
    train=False,
    augment=None
)

val_dataset = MultimodalDataset(
    val_df,
    train=False,
    augment=None
)

test_dataset = MultimodalDataset(
    test_df,
    train=False,
    augment=None
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

cal_loader = DataLoader(
    cal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Tamaño train_dataset:", len(train_dataset))
print("Tamaño train_eval_dataset original:", len(train_eval_dataset))
print("Tamaño cal_dataset:", len(cal_dataset))
print("Tamaño val_dataset:", len(val_dataset))
print("Tamaño test_dataset:", len(test_dataset))

model = OlmoEarthTiny_Multimodal_BiAttn_Residual(
    n_tabular=len(tabular_cols),
    n_image_channels=N_IMAGE_CHANNELS,
    n_thermal_channels=N_THERMAL_DERIVED_CHANNELS,
    pretrained=PRETRAINED_OLMOEARTH,
    d_model=512,
    n_heads=8,
    dropout=0.25,
    token_pool_size=(4, 4)
).to(DEVICE)

criterion = torch.nn.HuberLoss(delta=0.5)

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def get_new_modules(model):
    """
    Módulos nuevos, excluyendo OlmoEarth preentrenado.
    """
    return [
        model.spatial_attn,
        model.img_token_proj,
        model.img_token_encoder,
        model.img_vec_proj,
        model.img_token_ln,
        model.img_vec_ln,
        model.tab_mlp,
        model.tab_ln,
        model.thermal_stem,
        model.thermal_stage1,
        model.thermal_stage2,
        model.thermal_stage3,
        model.thermal_context,
        model.thermal_token_projs,
        model.thermal_token_encoder,
        model.thermal_ln,
        model.attn_img_to_tab,
        model.attn_tab_to_img,
        model.img_cross_fuse,
        model.tab_cross_fuse,
        model.concat_gate,
        model.se_fusion,
        model.head
    ]

def unfreeze_modules(modules):
    for module in modules:
        for p in module.parameters():
            p.requires_grad = True

def collect_params(modules):
    params = []
    seen = set()

    for module in modules:
        for p in module.parameters():
            if p.requires_grad and id(p) not in seen:
                params.append(p)
                seen.add(id(p))

    return params

def configure_stage(model, stage="head"):
    freeze_all(model)

    if stage == "head":
        # Entrena solo módulos nuevos; OlmoEarth congelado.
        unfreeze_modules(get_new_modules(model))

    elif stage == "backbone":
        # Fine-tuning suave de OlmoEarth + módulos nuevos.
        unfreeze_modules(get_new_modules(model))
        unfreeze_modules([model.backbone])

    elif stage == "full":
        # Fine-tuning completo.
        for p in model.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Stage no soportado: {stage}")

def build_optimizer(model, stage="head", hparams=None):
    hparams = hparams or {}
    lr_head = hparams.get("lr_head", LR_HEAD)
    lr_fine = hparams.get("lr_fine", LR_FINE)
    lr_full = hparams.get("lr_full", LR_FULL)
    weight_decay = hparams.get("weight_decay", WEIGHT_DECAY)

    new_modules = get_new_modules(model)

    if stage == "head":
        return torch.optim.AdamW(
            collect_params(new_modules),
            lr=lr_head,
            weight_decay=weight_decay
        )

    elif stage == "backbone":
        return torch.optim.AdamW(
            [
                {"params": collect_params([model.backbone]), "lr": lr_fine},
                {"params": collect_params(new_modules), "lr": lr_fine * 5.0},
            ],
            weight_decay=weight_decay
        )

    elif stage == "full":
        return torch.optim.AdamW(
            [
                {"params": collect_params([model.backbone]), "lr": lr_full},
                {"params": collect_params(new_modules), "lr": lr_full * 5.0},
            ],
            weight_decay=weight_decay
        )

    else:
        raise ValueError(f"Stage no soportado: {stage}")

def build_scheduler(optimizer, hparams=None):
    # Scheduler fijo: no forma parte de la búsqueda de Optuna.
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=SCHEDULER_MIN_LR
    )


In [ ]:
sample = MultimodalDataset(train_df, train=True, augment=train_augment)[0]

print("Optical shape:", sample["optical"].shape)
print("Thermal shape:", sample["thermal"].shape)
print("Tabular shape:", sample["tabular"].shape)
print("Target:", sample["target"])
print("ID:", sample["id"])
print("Source ID:", sample["source_id"])
print("Is synthetic:", sample["is_synthetic"])


In [ ]:
def inverse_target_transform(y_scaled, scaler, shift):
    y_scaled = np.array(y_scaled).reshape(-1, 1)
    y_log = scaler.inverse_transform(y_scaled).ravel()
    y = np.expm1(y_log)
    if shift != 0:
        y = y - shift
    return y

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return float("nan")

def compute_real_metrics(preds_scaled, targs_scaled):
    preds_real = inverse_target_transform(preds_scaled, target_scaler, target_shift)
    targs_real = inverse_target_transform(targs_scaled, target_scaler, target_shift)

    rmse_real = np.sqrt(np.mean((targs_real - preds_real) ** 2))
    mae_real = mean_absolute_error(targs_real, preds_real)
    r2_real = safe_r2(targs_real, preds_real)

    return {
        "preds_real": preds_real,
        "targets_real": targs_real,
        "rmse_real": rmse_real,
        "mae_real": mae_real,
        "r2_real": r2_real
    }

def _make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def _autocast_context():
    return torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=USE_AMP
    )


def evaluate_loader(model, loader, criterion):
    model.eval()
    preds_list, targs_list, ids_list, losses = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            optical = batch["optical"].to(DEVICE)
            thermal = batch["thermal"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE).float()
            targets = batch["target"].to(DEVICE).float()

            with _autocast_context():
                preds = model(optical, thermal, tabular)
                loss = criterion(preds, targets)

            losses.append(loss.item())
            preds_list.append(preds.cpu().numpy())
            targs_list.append(targets.cpu().numpy())
            ids_list.extend(batch["id"])

    preds_arr = np.concatenate(preds_list) if len(preds_list) > 0 else np.array([])
    targs_arr = np.concatenate(targs_list) if len(targs_list) > 0 else np.array([])

    if len(preds_arr) == 0:
        return {
            "loss": float("nan"),
            "rmse_scaled": float("nan"),
            "mae_scaled": float("nan"),
            "r2_scaled": float("nan"),
            "rmse_real": float("nan"),
            "mae_real": float("nan"),
            "r2_real": float("nan"),
            "preds": preds_arr,
            "targets": targs_arr,
            "preds_real": np.array([]),
            "targets_real": np.array([]),
            "ids": ids_list
        }

    rmse_scaled = np.sqrt(np.mean((targs_arr - preds_arr) ** 2))
    mae_scaled = mean_absolute_error(targs_arr, preds_arr)
    r2_scaled = safe_r2(targs_arr, preds_arr)

    real_metrics = compute_real_metrics(preds_arr, targs_arr)

    return {
        "loss": float(np.mean(losses)),
        "rmse_scaled": rmse_scaled,
        "mae_scaled": mae_scaled,
        "r2_scaled": r2_scaled,
        "rmse_real": real_metrics["rmse_real"],
        "mae_real": real_metrics["mae_real"],
        "r2_real": real_metrics["r2_real"],
        "preds": preds_arr,
        "targets": targs_arr,
        "preds_real": real_metrics["preds_real"],
        "targets_real": real_metrics["targets_real"],
        "ids": ids_list
    }

def get_current_lrs(optimizer):
    return [pg["lr"] for pg in optimizer.param_groups]

def train_phase(model, train_loader, train_eval_loader, val_loader, criterion,
                epochs, stage_name, ckpt_name, hparams=None, ckpt_dir=None,
                trial=None, step_offset=0):
    hparams = hparams or {}
    ckpt_dir = Path(ckpt_dir) if ckpt_dir is not None else OUT_DIR
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    configure_stage(model, stage_name)
    optimizer = build_optimizer(model, stage_name, hparams=hparams)
    scheduler = build_scheduler(optimizer, hparams=hparams)
    scaler = _make_grad_scaler()

    max_grad_norm = hparams.get("max_grad_norm", MAX_GRAD_NORM)
    best_val_rmse_real = np.inf
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        train_losses = []
        pbar = tqdm(train_loader, desc=f"{stage_name.capitalize()} Epoch {ep}", leave=False,dynamic_ncols=True, mininterval=0.5, position=0)

        for batch in pbar:
            optical = batch["optical"].to(DEVICE)
            thermal = batch["thermal"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE).float()
            targets = batch["target"].to(DEVICE).float()

            optimizer.zero_grad(set_to_none=True)

            with _autocast_context():
                preds = model(optical, thermal, tabular)
                loss = criterion(preds, targets)

            scaler.scale(loss).backward()

            trainable_params = [p for p in model.parameters() if p.requires_grad]
            if len(trainable_params) > 0 and max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)

            scaler.step(optimizer)
            scaler.update()

            train_losses.append(loss.item())
            pbar.set_postfix({"loss": f"{np.mean(train_losses):.4f}"}, refresh=True)

        train_metrics = evaluate_loader(model, train_eval_loader, criterion)
        val_metrics = evaluate_loader(model, val_loader, criterion)

        # El scheduler sigue la métrica real usada para seleccionar el mejor modelo
        scheduler.step(val_metrics["rmse_real"])

        current_lrs = ", ".join([f"{lr:.2e}" for lr in get_current_lrs(optimizer)])

        print(
            f"[{stage_name.upper()}] Epoch {ep}/{epochs} | "
            f"LR={current_lrs} | "
            f"TrainLoss={np.mean(train_losses):.4f} | "
            f"TrainRMSE_real={train_metrics['rmse_real']:.4f} | "
            f"TrainMAE_real={train_metrics['mae_real']:.4f} | "
            f"TrainR2_real={train_metrics['r2_real']:.4f}"
        )
        print(
            f"                 ValLoss={val_metrics['loss']:.4f} | "
            f"ValRMSE_real={val_metrics['rmse_real']:.4f} | "
            f"ValMAE_real={val_metrics['mae_real']:.4f} | "
            f"ValR2_real={val_metrics['r2_real']:.4f}"
        )

        history.append({
            "epoch": ep,
            "stage": stage_name,
            "train_loss": float(np.mean(train_losses)),
            "train_rmse_real": train_metrics["rmse_real"],
            "train_mae_real": train_metrics["mae_real"],
            "train_r2_real": train_metrics["r2_real"],
            "val_loss": val_metrics["loss"],
            "val_rmse_real": val_metrics["rmse_real"],
            "val_mae_real": val_metrics["mae_real"],
            "val_r2_real": val_metrics["r2_real"],
        })

        step = step_offset + ep
        if trial is not None:
            trial.report(val_metrics["rmse_real"], step=step)
            if trial.should_prune():
                raise optuna.TrialPruned(
                    f"Trial podado en {stage_name} epoch {ep}: "
                    f"ValRMSE_real={val_metrics['rmse_real']:.4f}"
                )

        # Guardar por métrica real de validación
        if val_metrics["rmse_real"] < best_val_rmse_real - 1e-6:
            best_val_rmse_real = val_metrics["rmse_real"]
            torch.save(
                {
                    "epoch": ep,
                    "stage": stage_name,
                    "model": model.state_dict(),
                    "opt": optimizer.state_dict(),
                    "val_loss": val_metrics["loss"],
                    "val_rmse_real": val_metrics["rmse_real"],
                    "val_mae_real": val_metrics["mae_real"],
                    "val_r2_real": val_metrics["r2_real"],
                    "hparams": hparams,
                },
                ckpt_dir / ckpt_name
            )
            print(f"  -> Mejor modelo ({stage_name}) guardado por ValRMSE_real.")

    return history


In [ ]:
## Cell 9: optimización de hiperparámetros con Optuna
def sample_hparams(trial):
    d_model = trial.suggest_categorical("d_model", [256, 384, 512])

    # n_heads condicionado a d_model para garantizar divisibilidad.
    if d_model == 384:
        n_heads = trial.suggest_categorical("n_heads_d384", [4, 6, 8])
    elif d_model == 256:
        n_heads = trial.suggest_categorical("n_heads_d256", [4, 8])
    else:
        n_heads = trial.suggest_categorical("n_heads_d512", [4, 8])

    # Cabeza regresora condicionada a d_model.
    if d_model == 256:
        head_hidden_1 = trial.suggest_categorical("head_hidden_1_d256", [384, 512, 768])
        head_hidden_2 = trial.suggest_categorical("head_hidden_2_d256", [128, 256, 384])
    elif d_model == 384:
        head_hidden_1 = trial.suggest_categorical("head_hidden_1_d384", [512, 768, 1024])
        head_hidden_2 = trial.suggest_categorical("head_hidden_2_d384", [192, 256, 384])
    else:
        head_hidden_1 = trial.suggest_categorical("head_hidden_1_d512", [768, 1024, 1536])
        head_hidden_2 = trial.suggest_categorical("head_hidden_2_d512", [256, 384, 512])

    token_pool_str = trial.suggest_categorical(
        "token_pool_size",
        ["2x2", "3x3", "4x4"]
    )
    token_pool_size = tuple(int(v) for v in token_pool_str.split("x"))

    img_ff_mult = trial.suggest_categorical("img_ff_mult", [2, 3, 4])
    thermal_ff_mult = trial.suggest_categorical("thermal_ff_mult", [2, 4])

    hparams = {
        # Arquitectura
        "d_model": d_model,
        "n_heads": n_heads,

        "dropout": trial.suggest_categorical(
            "dropout",
            [0.25, 0.30, 0.35, 0.40]
        ),

        "token_pool_size": token_pool_size,

        "img_token_layers": trial.suggest_categorical(
            "img_token_layers",
            [1, 2, 3]
        ),

        "thermal_token_layers": trial.suggest_categorical(
            "thermal_token_layers",
            [1, 2]
        ),

        "img_dim_feedforward": img_ff_mult * d_model,
        "thermal_dim_feedforward": thermal_ff_mult * d_model,

        "tab_hidden": trial.suggest_categorical(
            "tab_hidden",
            [128, 256, 512]
        ),

        "head_hidden_1": head_hidden_1,
        "head_hidden_2": head_hidden_2,

        "se_fusion_reduction": trial.suggest_categorical(
            "se_fusion_reduction",
            [4, 8, 16]
        ),

        "spatial_attention_kernel": trial.suggest_categorical(
            "spatial_attention_kernel",
            [3, 5, 7]
        ),

        # Optimización
        "lr_head": trial.suggest_categorical(
            "lr_head",
            [5e-5, 1e-4, 2e-4, 3e-4, 5e-4]
        ),

        "lr_fine": trial.suggest_categorical(
            "lr_fine",
            [1e-6, 3e-6, 5e-6, 1e-5, 2e-5, 5e-5]
        ),

        "lr_full": trial.suggest_categorical(
            "lr_full",
            [1e-7, 3e-7, 5e-7, 1e-6, 3e-6]
        ),

        "weight_decay": trial.suggest_categorical(
            "weight_decay",
            [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 5e-4]
        ),

        "max_grad_norm": trial.suggest_categorical(
            "max_grad_norm",
            [0.5, 1.0, 2.0, 3.0, 5.0]
        ),

        "huber_delta": trial.suggest_categorical(
            "huber_delta",
            [0.25, 0.50, 0.75, 1.00, 1.50]
        ),

        # Duración fija del entrenamiento por fases
        "epochs_head": EPOCHS_HEAD,
        "epochs_backbone": EPOCHS_BACKBONE,
        "epochs_full": EPOCHS_FULL,

        # Datos / balanceo
        "batch_size": trial.suggest_categorical(
            "batch_size",
            [4, 8, 12, 16]
        ),

        "wsmoter_multiplier": trial.suggest_categorical(
            "wsmoter_multiplier",
            [1.1, 1.2, 1.3, 1.4]
        ),

        "wsmoter_weight_power": trial.suggest_categorical(
            "wsmoter_weight_power",
            [1.0, 1.5, 2.0, 2.5, 3.0]
        ),

        "lds_sigma": trial.suggest_categorical(
            "lds_sigma",
            [3.0, 5.0, 7.0, 10.0, 12.0]
        ),

        # Aumentación
        "use_image_augmentation": trial.suggest_categorical(
            "use_image_augmentation",
            [True, False]
        ),

        "aug_p_hflip": trial.suggest_categorical(
            "aug_p_hflip",
            [0.0, 0.25, 0.35, 0.45]
        ),

        "aug_p_vflip": trial.suggest_categorical(
            "aug_p_vflip",
            [0.0, 0.25, 0.35, 0.45]
        ),

        "aug_p_rot90": trial.suggest_categorical(
            "aug_p_rot90",
            [0.0, 0.25, 0.35, 0.45]
        ),

        "aug_p_noise": trial.suggest_categorical(
            "aug_p_noise",
            [0.0, 0.20, 0.35, 0.50, 0.60]
        ),

        "aug_noise_sigma_image": trial.suggest_categorical(
            "aug_noise_sigma_image",
            [0.005, 0.010, 0.020, 0.025, 0.035, 0.050]
        ),

        "aug_noise_sigma_thermal": trial.suggest_categorical(
            "aug_noise_sigma_thermal",
            [0.005, 0.010, 0.020, 0.025, 0.035, 0.050]
        ),
    }

    if not (hparams["lr_full"] <= hparams["lr_fine"] <= hparams["lr_head"]):
        raise optuna.TrialPruned(
            "Se poda porque no cumple lr_full <= lr_fine <= lr_head"
        )

    return hparams


def _jsonable_hparams(hparams):
    out = dict(hparams)
    if isinstance(out.get("token_pool_size"), tuple):
        out["token_pool_size"] = list(out["token_pool_size"])
    return out


def prepare_dataframes_for_hparams(hparams, include_test=False, persist=False):
    """
    Reconstruye los dataframes desde los splits crudos.
    En Optuna se llama con include_test=False para no tocar test dentro del objective.
    """
    global target_scaler, target_shift
    global train_df, train_original_df, cal_df, val_df, test_df

    train_base = train_raw_split_df.reset_index(drop=True).copy()
    cal_base = cal_raw_split_df.reset_index(drop=True).copy()
    val_base = val_raw_split_df.reset_index(drop=True).copy()
    test_base = test_raw_split_df.reset_index(drop=True).copy() if include_test else None

    # Pesos LDS solo en train.
    train_base = add_gradient_weights_lds(
        train_base,
        target_col=TARGET_COL,
        weight_col="Gradient Weight",
        sigma=hparams["lds_sigma"]
    )

    # Escalado tabular ajustado solo con train original.
    scaler_tab_local = StandardScaler()
    train_base[scale_cols] = scaler_tab_local.fit_transform(train_base[scale_cols])
    cal_base[scale_cols] = scaler_tab_local.transform(cal_base[scale_cols])
    val_base[scale_cols] = scaler_tab_local.transform(val_base[scale_cols])
    if include_test:
        test_base[scale_cols] = scaler_tab_local.transform(test_base[scale_cols])

    train_original_local = train_base.copy()

    if USE_WSMOTER_TABULAR:
        train_local = wsmoter_oversample_regression(
            df=train_base,
            feature_cols=WSMOTER_FEATURE_COLS,
            target_col=TARGET_COL,
            id_col="ID",
            weight_col="Gradient Weight",
            multiplier=hparams["wsmoter_multiplier"],
            k_neighbors=WSMOTER_K_NEIGHBORS,
            weight_power=hparams["wsmoter_weight_power"],
            random_state=SEED,
            id_mode=WSMOTER_ID_MODE
        )
    else:
        train_local = train_base.copy()
        train_local["is_synthetic"] = 0
        train_local["source_ID"] = train_local["ID"].astype(int)

    cal_local = cal_base.copy()
    cal_local["is_synthetic"] = 0
    cal_local["source_ID"] = cal_local["ID"].astype(int)
    cal_local = cal_local.drop(columns=["Gradient Weight"], errors="ignore")

    val_local = val_base.copy()
    val_local["is_synthetic"] = 0
    val_local["source_ID"] = val_local["ID"].astype(int)
    val_local = val_local.drop(columns=["Gradient Weight"], errors="ignore")

    train_original_local["is_synthetic"] = 0
    train_original_local["source_ID"] = train_original_local["ID"].astype(int)

    if include_test:
        test_local = test_base.copy()
        test_local["is_synthetic"] = 0
        test_local["source_ID"] = test_local["ID"].astype(int)
        test_local = test_local.drop(columns=["Gradient Weight"], errors="ignore")
    else:
        test_local = None

    dataframes_for_target = [train_local, train_original_local, cal_local, val_local]
    if include_test:
        dataframes_for_target.append(test_local)

    for _df in dataframes_for_target:
        _df[TARGET_COL + "_raw"] = _df[TARGET_COL].copy()

    min_train_target = train_original_local[TARGET_COL].min()
    target_shift_local = 0.0

    if min_train_target <= -1:
        target_shift_local = float(abs(min_train_target) + 1.0)
        for _df in dataframes_for_target:
            _df[TARGET_COL] += target_shift_local

    for _df in dataframes_for_target:
        _df[TARGET_COL] = np.log1p(_df[TARGET_COL].astype(float))

    scaler_y_local = StandardScaler()
    train_local[[TARGET_COL]] = scaler_y_local.fit_transform(train_local[[TARGET_COL]])
    train_original_local[[TARGET_COL]] = scaler_y_local.transform(train_original_local[[TARGET_COL]])
    cal_local[[TARGET_COL]] = scaler_y_local.transform(cal_local[[TARGET_COL]])
    val_local[[TARGET_COL]] = scaler_y_local.transform(val_local[[TARGET_COL]])
    if include_test:
        test_local[[TARGET_COL]] = scaler_y_local.transform(test_local[[TARGET_COL]])

    target_scaler = scaler_y_local
    target_shift = target_shift_local

    if persist:
        joblib.dump(scaler_tab_local, OUT_DIR / "tabular_scaler_best_optuna.pkl")
        joblib.dump(scaler_y_local, OUT_DIR / "target_scaler_best_optuna.pkl")
        joblib.dump({"shift": target_shift_local}, OUT_DIR / "target_shift_best_optuna.json")
        train_local.to_csv(OUT_DIR / "train_wsmoter_best_optuna.csv", index=False)
        train_original_local.to_csv(OUT_DIR / "train_original_best_optuna.csv", index=False)
        cal_local.to_csv(OUT_DIR / "calibration_best_optuna.csv", index=False)
        val_local.to_csv(OUT_DIR / "val_best_optuna.csv", index=False)
        if include_test:
            test_local.to_csv(OUT_DIR / "test_best_optuna.csv", index=False)

    return train_local, train_original_local, cal_local, val_local, test_local, scaler_y_local, target_shift_local


def build_loaders_for_hparams(train_local, train_original_local, cal_local, val_local, test_local, hparams):
    train_augment_local = MultibandPairAugment(
        p_hflip=hparams["aug_p_hflip"],
        p_vflip=hparams["aug_p_vflip"],
        p_rot90=hparams["aug_p_rot90"],
        p_noise=hparams["aug_p_noise"],
        noise_sigma_image=hparams["aug_noise_sigma_image"],
        noise_sigma_thermal=hparams["aug_noise_sigma_thermal"],
        clip_value=AUG_CLIP_VALUE
    ) if hparams["use_image_augmentation"] else None

    train_dataset_local = MultimodalDataset(train_local, train=True, augment=train_augment_local)
    train_eval_dataset_local = MultimodalDataset(train_original_local, train=False, augment=None)
    cal_dataset_local = MultimodalDataset(cal_local, train=False, augment=None)
    val_dataset_local = MultimodalDataset(val_local, train=False, augment=None)

    batch_size = hparams["batch_size"]
    common = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

    loaders = {
        "train": DataLoader(
            train_dataset_local,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
            **common
        ),
        "train_eval": DataLoader(
            train_eval_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        ),
        "calibration": DataLoader(
            cal_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        ),
        "val": DataLoader(
            val_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        ),
        "test": None
    }

    if test_local is not None:
        test_dataset_local = MultimodalDataset(test_local, train=False, augment=None)
        loaders["test"] = DataLoader(
            test_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        )

    return loaders


def build_model_from_hparams(hparams):
    return OlmoEarthTiny_Multimodal_BiAttn_Residual(
        n_tabular=len(tabular_cols),
        n_image_channels=N_IMAGE_CHANNELS,
        n_thermal_channels=N_THERMAL_DERIVED_CHANNELS,
        pretrained=PRETRAINED_OLMOEARTH,
        d_model=hparams["d_model"],
        n_heads=hparams["n_heads"],
        dropout=hparams["dropout"],
        token_pool_size=tuple(hparams["token_pool_size"]),
        img_token_layers=hparams["img_token_layers"],
        thermal_token_layers=hparams["thermal_token_layers"],
        img_dim_feedforward=hparams["img_dim_feedforward"],
        thermal_dim_feedforward=hparams["thermal_dim_feedforward"],
        tab_hidden=hparams["tab_hidden"],
        head_hidden_1=hparams["head_hidden_1"],
        head_hidden_2=hparams["head_hidden_2"],
        se_fusion_reduction=hparams["se_fusion_reduction"],
        spatial_attention_kernel=hparams["spatial_attention_kernel"]
    ).to(DEVICE)


def find_best_checkpoint(ckpt_dir):
    candidates = []
    for name in ["best_head.pth", "best_backbone.pth", "best_full.pth"]:
        path = Path(ckpt_dir) / name
        if path.exists():
            ck = torch.load(path, map_location="cpu", weights_only=False)
            candidates.append((float(ck.get("val_rmse_real", np.inf)), path, ck.get("stage", name)))

    if len(candidates) == 0:
        raise FileNotFoundError(f"No se encontró ningún checkpoint en {ckpt_dir}")

    candidates = sorted(candidates, key=lambda x: x[0])
    return candidates[0][1], candidates[0][0], candidates[0][2]


def fit_staged_model(model, loaders, criterion, hparams, ckpt_dir, trial=None):
    history_all = []
    step_offset = 0

    history_head = train_phase(
        model=model,
        train_loader=loaders["train"],
        train_eval_loader=loaders["train_eval"],
        val_loader=loaders["val"],
        criterion=criterion,
        epochs=hparams["epochs_head"],
        stage_name="head",
        ckpt_name="best_head.pth",
        hparams=hparams,
        ckpt_dir=ckpt_dir,
        trial=trial,
        step_offset=step_offset
    )
    history_all.extend(history_head)
    step_offset += hparams["epochs_head"]

    ck = torch.load(Path(ckpt_dir) / "best_head.pth", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])

    history_backbone = train_phase(
        model=model,
        train_loader=loaders["train"],
        train_eval_loader=loaders["train_eval"],
        val_loader=loaders["val"],
        criterion=criterion,
        epochs=hparams["epochs_backbone"],
        stage_name="backbone",
        ckpt_name="best_backbone.pth",
        hparams=hparams,
        ckpt_dir=ckpt_dir,
        trial=trial,
        step_offset=step_offset
    )
    history_all.extend(history_backbone)
    step_offset += hparams["epochs_backbone"]

    if hparams["epochs_full"] > 0:
        best_so_far_path, _, _ = find_best_checkpoint(ckpt_dir)
        ck = torch.load(best_so_far_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"])

        history_full = train_phase(
            model=model,
            train_loader=loaders["train"],
            train_eval_loader=loaders["train_eval"],
            val_loader=loaders["val"],
            criterion=criterion,
            epochs=hparams["epochs_full"],
            stage_name="full",
            ckpt_name="best_full.pth",
            hparams=hparams,
            ckpt_dir=ckpt_dir,
            trial=trial,
            step_offset=step_offset
        )
        history_all.extend(history_full)

    best_path, best_val_rmse, best_stage = find_best_checkpoint(ckpt_dir)
    pd.DataFrame(history_all).to_csv(Path(ckpt_dir) / "history.csv", index=False)

    return history_all, best_path, best_val_rmse, best_stage


def objective(trial):
    global target_scaler, target_shift

    hparams = sample_hparams(trial)
    trial.set_user_attr("resolved_hparams", _jsonable_hparams(hparams))

    trial_dir = OPTUNA_TMP_DIR / f"trial_{trial.number:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)

    model_trial = None
    loaders_trial = None

    try:
        seed_everything(SEED + trial.number)

        train_local, train_original_local, cal_local, val_local, _, scaler_y_local, shift_local = prepare_dataframes_for_hparams(
            hparams,
            include_test=False,
            persist=False
        )

        loaders_trial = build_loaders_for_hparams(
            train_local=train_local,
            train_original_local=train_original_local,
            cal_local=cal_local,
            val_local=val_local,
            test_local=None,
            hparams=hparams
        )

        model_trial = build_model_from_hparams(hparams)
        criterion_trial = torch.nn.HuberLoss(delta=hparams["huber_delta"])

        _, best_path, best_val_rmse, best_stage = fit_staged_model(
            model=model_trial,
            loaders=loaders_trial,
            criterion=criterion_trial,
            hparams=hparams,
            ckpt_dir=trial_dir,
            trial=trial
        )

        trial.set_user_attr("best_checkpoint", str(best_path))
        trial.set_user_attr("best_stage", best_stage)
        trial.set_user_attr("target_shift", shift_local)

        return best_val_rmse

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            raise optuna.TrialPruned("CUDA out of memory")
        raise

    finally:
        del model_trial
        del loaders_trial
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

class KeepTopKCheckpoints:
    def __init__(self, top_k, top_k_dir, tmp_dir):
        self.top_k = int(top_k)
        self.top_k_dir = Path(top_k_dir)
        self.tmp_dir = Path(tmp_dir)
        self.top_k_dir.mkdir(parents=True, exist_ok=True)

    def _read_ckpt_metadata(self, ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)

        stage = ck.get("stage", "unknown")
        epoch = ck.get("epoch", "unknown")
        rmse = ck.get("val_rmse_real", None)

        # Si tus épocas son 0-indexed y quieres nombres humanos 1-indexed,
        # cambia esto a: epoch = int(epoch) + 1
        if isinstance(epoch, int):
            epoch_str = f"{epoch:03d}"
        else:
            epoch_str = str(epoch)

        return stage, epoch_str, rmse

    def __call__(self, study, frozen_trial):
        trial_dir = self.tmp_dir / f"trial_{frozen_trial.number:03d}"

        try:
            complete_trials = [
                t for t in study.trials
                if t.state == optuna.trial.TrialState.COMPLETE
                and t.value is not None
            ]

            complete_trials = sorted(complete_trials, key=lambda t: float(t.value))
            top_trials = complete_trials[:self.top_k]
            top_trial_numbers = {t.number for t in top_trials}

            # Copiar el checkpoint del trial actual si quedó dentro del top K.
            if (
                frozen_trial.state == optuna.trial.TrialState.COMPLETE
                and frozen_trial.value is not None
                and frozen_trial.number in top_trial_numbers
            ):
                best_ckpt = frozen_trial.user_attrs.get("best_checkpoint")

                if best_ckpt is not None and Path(best_ckpt).exists():
                    best_ckpt = Path(best_ckpt)

                    stage, epoch_str, ckpt_rmse = self._read_ckpt_metadata(best_ckpt)

                    # Usa el RMSE del checkpoint si existe; si no, usa el valor del trial.
                    value = float(ckpt_rmse) if ckpt_rmse is not None else float(frozen_trial.value)

                    dst_name = (
                        f"trial_{frozen_trial.number:03d}"
                        f"_stage_{stage}"
                        f"_epoch_{epoch_str}"
                        f"_rmse_{value:.6f}.pth"
                    )

                    dst_path = self.top_k_dir / dst_name

                    # Eliminar versiones previas del mismo trial.
                    for old in self.top_k_dir.glob(f"trial_{frozen_trial.number:03d}_stage_*_epoch_*_rmse_*.pth"):
                        old.unlink(missing_ok=True)

                    shutil.copy2(best_ckpt, dst_path)

                    print(f"[Optuna] Checkpoint top-{self.top_k} guardado: {dst_path}")

            # Eliminar checkpoints persistentes de trials que ya salieron del top K.
            for ckpt_path in self.top_k_dir.glob("trial_*_stage_*_epoch_*_rmse_*.pth"):
                try:
                    trial_num = int(ckpt_path.name.split("_")[1])
                except Exception:
                    continue

                if trial_num not in top_trial_numbers:
                    ckpt_path.unlink(missing_ok=True)
                    print(f"[Optuna] Eliminado del top-{self.top_k}: {ckpt_path}")

            # Crear manifiesto actualizado.
            manifest = []

            for rank, t in enumerate(top_trials, start=1):
                matches = list(
                    self.top_k_dir.glob(
                        f"trial_{t.number:03d}_stage_*_epoch_*_rmse_*.pth"
                    )
                )

                checkpoint_path = str(matches[0]) if matches else None

                manifest.append({
                    "rank": rank,
                    "trial": int(t.number),
                    "value": float(t.value),
                    "checkpoint": checkpoint_path,
                    "best_stage": t.user_attrs.get("best_stage"),
                    "resolved_hparams": t.user_attrs.get("resolved_hparams"),
                })

            with open(self.top_k_dir / "top5_manifest.json", "w", encoding="utf-8") as fh:
                json.dump(manifest, fh, indent=2, ensure_ascii=False)

        finally:
            # Borra siempre la carpeta temporal del trial actual.
            if trial_dir.exists():
                shutil.rmtree(trial_dir, ignore_errors=True)

if USE_OPTUNA:
    sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

    study = optuna.create_study(
        direction="minimize",
        study_name=OPTUNA_STUDY_NAME,
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        objective,
        n_trials=OPTUNA_N_TRIALS,
        timeout=OPTUNA_TIMEOUT,
        gc_after_trial=True,
        show_progress_bar=True,
        callbacks=[
        KeepTopKCheckpoints(
            top_k=TOP_K,
            top_k_dir=TOP_K_CKPT_DIR,
            tmp_dir=OPTUNA_TMP_DIR
        )
        ]
    )

    print("\n===== MEJOR TRIAL OPTUNA =====")
    print("Trial:", study.best_trial.number)
    print("ValRMSE_real:", study.best_value)
    print("Mejor stage:", study.best_trial.user_attrs.get("best_stage"))
    print("Checkpoint:", study.best_trial.user_attrs.get("best_checkpoint"))
    print("Hiperparámetros resueltos:")
    print(json.dumps(study.best_trial.user_attrs["resolved_hparams"], indent=2, ensure_ascii=False))

    study.trials_dataframe().to_csv(OUT_DIR / "optuna_trials.csv", index=False)
    with open(OUT_DIR / "best_hparams_optuna.json", "w", encoding="utf-8") as fh:
        json.dump(study.best_trial.user_attrs["resolved_hparams"], fh, indent=2, ensure_ascii=False)
else:
    print("USE_OPTUNA=False. Se omite la búsqueda de hiperparámetros.")

In [ ]:
# Cell 10: evaluación final con el mejor modelo de Optuna

if not USE_OPTUNA:
    raise RuntimeError("Activa USE_OPTUNA=True para usar esta celda.")

best_hparams = dict(study.best_trial.user_attrs["resolved_hparams"])
best_hparams["token_pool_size"] = tuple(best_hparams["token_pool_size"])

def resolve_best_optuna_checkpoint(study):
    """
    Busca el checkpoint real del mejor trial.
    Primero intenta la ruta original.
    Si fue borrada de /kaggle/temp, busca en TOP_K_CKPT_DIR.
    """
    best_trial_num = study.best_trial.number

    # 1. Ruta original guardada por Optuna
    original_path = Path(study.best_trial.user_attrs["best_checkpoint"])
    if original_path.exists():
        return original_path

    print("Checkpoint temporal no encontrado:")
    print(original_path)
    print("Buscando checkpoint persistente en TOP_K_CKPT_DIR...")

    # 2. Buscar en el manifiesto top5
    manifest_path = TOP_K_CKPT_DIR / "top5_manifest.json"

    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as fh:
            manifest = json.load(fh)

        for row in manifest:
            if int(row["trial"]) == int(best_trial_num):
                ckpt = row.get("checkpoint", None)
                if ckpt is not None:
                    ckpt = Path(ckpt)
                    if ckpt.exists():
                        print("Checkpoint encontrado desde manifest:")
                        print(ckpt)
                        return ckpt

    # 3. Buscar directamente por nombre de trial
    matches = sorted(
        TOP_K_CKPT_DIR.glob(f"trial_{best_trial_num:03d}_stage_*_epoch_*_rmse_*.pth")
    )

    if len(matches) > 0:
        def get_rmse_from_name(path):
            m = re.search(r"_rmse_([0-9.]+)\.pth$", path.name)
            return float(m.group(1)) if m else float("inf")

        best_match = min(matches, key=get_rmse_from_name)
        print("Checkpoint encontrado por búsqueda directa:")
        print(best_match)
        return best_match

    # 4. Diagnóstico si no aparece
    available_ckpts = list(TOP_K_CKPT_DIR.glob("*.pth"))

    msg = (
        f"No se encontró checkpoint para el mejor trial {best_trial_num}.\n\n"
        f"Ruta original borrada:\n{original_path}\n\n"
        f"Carpeta persistente revisada:\n{TOP_K_CKPT_DIR}\n\n"
        f"Checkpoints disponibles:\n"
    )

    if len(available_ckpts) == 0:
        msg += "No hay checkpoints .pth en TOP_K_CKPT_DIR."
    else:
        msg += "\n".join(str(p) for p in available_ckpts)

    raise FileNotFoundError(msg)


best_ckpt_path = resolve_best_optuna_checkpoint(study)

# Opcional pero recomendado: copiar el mejor checkpoint a una ruta fija
persistent_best_ckpt = OUT_DIR / "best_checkpoint_optuna.pth"

if best_ckpt_path != persistent_best_ckpt:
    shutil.copy2(best_ckpt_path, persistent_best_ckpt)
    best_ckpt_path = persistent_best_ckpt

print("Checkpoint final que se usará:")
print(best_ckpt_path)

# Reconstruir exactamente los dataframes del mejor trial, ahora incluyendo test.
train_df, train_original_df, cal_df, val_df, test_df, target_scaler, target_shift = prepare_dataframes_for_hparams(
    best_hparams,
    include_test=True,
    persist=True
)

loaders_best = build_loaders_for_hparams(
    train_local=train_df,
    train_original_local=train_original_df,
    cal_local=cal_df,
    val_local=val_df,
    test_local=test_df,
    hparams=best_hparams
)

train_loader = loaders_best["train"]
train_eval_loader = loaders_best["train_eval"]
cal_loader = loaders_best["calibration"]
val_loader = loaders_best["val"]
test_loader = loaders_best["test"]

model = build_model_from_hparams(best_hparams)
criterion = torch.nn.HuberLoss(delta=best_hparams["huber_delta"])

ck = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"])
print("Checkpoint Optuna cargado:", best_ckpt_path)
print("Stage:", ck.get("stage", "unk"), "epoch:", ck.get("epoch", "unk"))
print("ValRMSE_real guardado:", ck.get("val_rmse_real", "unk"))

# TRAIN ORIGINAL, sin WSMOTER y sin augmentación
train_eval = evaluate_loader(model, train_eval_loader, criterion)
train_preds_real = train_eval["preds_real"]
train_targs_real = train_eval["targets_real"]
ids_train = train_eval["ids"]

train_rmse = train_eval["rmse_real"]
train_mae = train_eval["mae_real"]
train_r2 = train_eval["r2_real"]

# CALIBRACIÓN CONFORMAL: no se usa para entrenar ni seleccionar hiperparámetros.
cal_eval = evaluate_loader(model, cal_loader, criterion)
cal_preds_real = cal_eval["preds_real"]
cal_targs_real = cal_eval["targets_real"]
ids_cal = cal_eval["ids"]

# VALIDACIÓN
val_eval = evaluate_loader(model, val_loader, criterion)
val_preds_real = val_eval["preds_real"]
val_targs_real = val_eval["targets_real"]
ids_val = val_eval["ids"]

val_rmse = val_eval["rmse_real"]
val_mae = val_eval["mae_real"]
val_r2 = val_eval["r2_real"]

# TEST: se usa solo después de elegir los mejores hiperparámetros.
test_eval = evaluate_loader(model, test_loader, criterion)
test_preds_real = test_eval["preds_real"]
test_targs_real = test_eval["targets_real"]
ids_test = test_eval["ids"]

test_rmse = test_eval["rmse_real"]
test_mae = test_eval["mae_real"]
test_r2 = test_eval["r2_real"]

print("\n===== TRAIN ORIGINAL, sin sintéticos (unidades reales: °C/Km) =====")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (train_rmse, train_mae, train_r2))

print("\n===== VALIDATION (unidades reales: °C/Km) =====")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (val_rmse, val_mae, val_r2))

print("\n===== TEST FINAL OPTUNA (unidades reales: °C/Km) =====")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (test_rmse, test_mae, test_r2))

final_metrics = {
    "train_rmse": float(train_rmse),
    "train_mae": float(train_mae),
    "train_r2": float(train_r2),
    "val_rmse": float(val_rmse),
    "val_mae": float(val_mae),
    "val_r2": float(val_r2),
    "test_rmse": float(test_rmse),
    "test_mae": float(test_mae),
    "test_r2": float(test_r2),
    "best_checkpoint": str(best_ckpt_path),
    "best_hparams": _jsonable_hparams(best_hparams)
}

with open(OUT_DIR / "final_metrics_best_optuna.json", "w", encoding="utf-8") as fh:
    json.dump(final_metrics, fh, indent=2, ensure_ascii=False)


## Split Conformal Prediction

Esta sección usa exclusivamente el conjunto **calibration** para calcular el score conformal.  
El modelo y sus hiperparámetros ya fueron fijados usando **train + validation**; `calibration` y `test` permanecen fuera de ese proceso.

El score se calcula en el **espacio transformado/estandarizado del target**:

\[
s_i = |y_i^{(s)} - \hat y_i^{(s)}|.
\]

Luego se obtiene el cuantil conformal con corrección finita y se construyen los límites en ese mismo espacio. Finalmente se aplica la transformación inversa, por lo que los intervalos se reportan en **°C/km** y pueden ser asimétricos en la escala real.


In [ ]:
# Split Conformal Prediction sobre el MEJOR modelo de Optuna

def conformal_quantile(scores, alpha=0.10):
    """
    Cuantil split-conformal con corrección de muestra finita:
        q_level = ceil((n + 1) * (1 - alpha)) / n
    usando el método 'higher'.
    """
    scores = np.asarray(scores, dtype=float).ravel()
    scores = scores[np.isfinite(scores)]

    if len(scores) == 0:
        raise ValueError("No hay scores de calibración válidos.")

    n = len(scores)
    q_level = np.ceil((n + 1) * (1.0 - alpha)) / n
    q_level = min(1.0, float(q_level))

    try:
        qhat = np.quantile(scores, q_level, method="higher")
    except TypeError:
        qhat = np.quantile(scores, q_level, interpolation="higher")

    return float(qhat), float(q_level)


def build_conformal_intervals(cal_eval, test_eval, alpha=CONFORMAL_ALPHA):
    # Scores en el espacio exacto donde el modelo fue entrenado.
    cal_scores = np.abs(
        np.asarray(cal_eval["targets"]).ravel()
        - np.asarray(cal_eval["preds"]).ravel()
    )

    qhat, q_level = conformal_quantile(cal_scores, alpha=alpha)

    test_pred_scaled = np.asarray(test_eval["preds"]).ravel()
    lower_scaled = test_pred_scaled - qhat
    upper_scaled = test_pred_scaled + qhat

    # Transformación monótona de vuelta a °C/km.
    lower_real = inverse_target_transform(
        lower_scaled,
        target_scaler,
        target_shift
    )
    upper_real = inverse_target_transform(
        upper_scaled,
        target_scaler,
        target_shift
    )

    pred_real = np.asarray(test_eval["preds_real"]).ravel()
    true_real = np.asarray(test_eval["targets_real"]).ravel()
    ids = np.asarray(test_eval["ids"], dtype=int)

    covered = (true_real >= lower_real) & (true_real <= upper_real)
    width = upper_real - lower_real

    results = pd.DataFrame({
        "ID": ids,
        "target_real": true_real,
        "prediction_real": pred_real,
        "lower_conformal": lower_real,
        "upper_conformal": upper_real,
        "interval_width": width,
        "covered": covered.astype(int),
        "absolute_error": np.abs(true_real - pred_real),
    })

    metrics = {
        "alpha": float(alpha),
        "nominal_coverage": float(1.0 - alpha),
        "n_calibration": int(len(cal_scores)),
        "q_level": float(q_level),
        "qhat_scaled": float(qhat),
        "empirical_test_coverage": float(np.mean(covered)),
        "mean_interval_width_real": float(np.mean(width)),
        "median_interval_width_real": float(np.median(width)),
        "min_interval_width_real": float(np.min(width)),
        "max_interval_width_real": float(np.max(width)),
    }

    return results, metrics, cal_scores


conformal_test_df, conformal_metrics, conformal_cal_scores = build_conformal_intervals(
    cal_eval=cal_eval,
    test_eval=test_eval,
    alpha=CONFORMAL_ALPHA
)

# Coordenadas para análisis cartográfico.
coords_for_conformal = df_final[
    ["ID", GEO_LAT_COL, GEO_LON_COL, PROJECTED_X_COL, PROJECTED_Y_COL]
].copy()
coords_for_conformal["ID"] = coords_for_conformal["ID"].astype(int)

conformal_test_df = coords_for_conformal.merge(
    conformal_test_df,
    on="ID",
    how="right"
)

conformal_csv = OUT_DIR / "conformal_test_intervals.csv"
conformal_json = OUT_DIR / "conformal_metrics.json"
conformal_scores_csv = OUT_DIR / "conformal_calibration_scores.csv"

conformal_test_df.to_csv(conformal_csv, index=False)
pd.DataFrame({"calibration_score_scaled": conformal_cal_scores}).to_csv(
    conformal_scores_csv,
    index=False
)

# Cobertura por cuartiles del target real: diagnóstico de cobertura condicional aproximada.
quartile_df = conformal_test_df.copy()
quartile_df["target_quartile"] = pd.qcut(
    quartile_df["target_real"],
    q=4,
    duplicates="drop"
)
conformal_by_quartile = (
    quartile_df
    .groupby("target_quartile", observed=True)
    .agg(
        n=("ID", "size"),
        empirical_coverage=("covered", "mean"),
        mean_interval_width=("interval_width", "mean"),
        mean_absolute_error=("absolute_error", "mean")
    )
    .reset_index()
)
conformal_by_quartile["target_quartile"] = conformal_by_quartile["target_quartile"].astype(str)
conformal_quartile_csv = OUT_DIR / "conformal_coverage_by_target_quartile.csv"
conformal_by_quartile.to_csv(conformal_quartile_csv, index=False)

with open(conformal_json, "w", encoding="utf-8") as fh:
    json.dump(conformal_metrics, fh, indent=2, ensure_ascii=False)

print("\n===== SPLIT CONFORMAL PREDICTION =====")
print(f"Cobertura nominal: {100*conformal_metrics['nominal_coverage']:.1f}%")
print(f"Cobertura empírica TEST: {100*conformal_metrics['empirical_test_coverage']:.1f}%")
print(f"n calibración: {conformal_metrics['n_calibration']}")
print(f"qhat (espacio escalado): {conformal_metrics['qhat_scaled']:.5f}")
print(f"Ancho medio intervalo: {conformal_metrics['mean_interval_width_real']:.3f} °C/km")
print(f"Ancho mediano intervalo: {conformal_metrics['median_interval_width_real']:.3f} °C/km")

# Figura 1: distribución de scores de calibración + qhat
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(conformal_cal_scores, bins=20, alpha=0.80, edgecolor="black")
ax.axvline(
    conformal_metrics["qhat_scaled"],
    linestyle="--",
    linewidth=2,
    label=f"qhat={conformal_metrics['qhat_scaled']:.3f}"
)
ax.set_xlabel("Score conformal |y - ŷ| (espacio escalado)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de scores de calibración")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "conformal_calibration_scores.pdf", dpi=300, bbox_inches="tight")
plt.show()

# Figura 2: intervalos conformales en TEST ordenados por target real
plot_df = conformal_test_df.sort_values("target_real").reset_index(drop=True)
x = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(
    x,
    plot_df["lower_conformal"].to_numpy(),
    plot_df["upper_conformal"].to_numpy(),
    alpha=0.25,
    label=f"Intervalo conformal {100*(1-CONFORMAL_ALPHA):.0f}%"
)
ax.plot(x, plot_df["prediction_real"], linewidth=1.4, label="Predicción")
ax.scatter(x, plot_df["target_real"], s=18, label="Real", zorder=3)
ax.set_xlabel("Observaciones TEST ordenadas por gradiente real")
ax.set_ylabel("Gradiente geotérmico (°C/km)")
ax.set_title(
    f"Split Conformal Prediction — cobertura TEST "
    f"{100*conformal_metrics['empirical_test_coverage']:.1f}%"
)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "conformal_test_intervals.pdf", dpi=300, bbox_inches="tight")
plt.show()

print("Guardado:", conformal_csv)
print("Guardado:", conformal_json)
print("Guardado:", conformal_scores_csv)
print("Guardado:", conformal_quartile_csv)
print("\nCobertura por cuartiles del target:")
display(conformal_by_quartile)


## Exportación de atención espacial y atención cruzada

Se conservan dos mecanismos interpretables:

1. **SpatialAttention**: mapa escalar aprendido sobre los embeddings espaciales de OlmoEarth.
2. **Cross-attention tabla → imagen**: muestra qué tokens espaciales de la imagen reciben mayor peso cuando la consulta proviene de la representación tabular.

> Nota: la dirección **imagen → tabla** tiene una sola clave tabular en esta arquitectura. Por ello su softmax se calcula sobre una única posición y sus pesos son trivialmente 1. Para interpretación espacial, la dirección informativa es **tabla → imagen**.


In [ ]:
# ============================================================
# Exportar atención para el artículo
# ============================================================

def collect_attention_information(model, loader, split_name="test"):
    model.eval()

    ids_all = []
    preds_scaled_all = []
    targets_scaled_all = []
    spatial_all = []
    tab_to_img_heads_all = []
    img_to_tab_heads_all = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Attention export: {split_name}"):
            optical = batch["optical"].to(DEVICE)
            thermal = batch["thermal"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE).float()
            targets = batch["target"].to(DEVICE).float()

            # Para interpretabilidad se usa inferencia float32.
            preds, attn = model(
                optical,
                thermal,
                tabular,
                return_attention=True
            )

            spatial = attn["spatial"].detach().float().cpu().numpy()[:, 0]

            # [B, heads, 1, N_image] -> [B, heads, N_image]
            tab_to_img = attn["tab_to_img"].detach().float().cpu().numpy()[:, :, 0, :]

            # [B, heads, N_image, 1] -> [B, heads, N_image]
            # En esta arquitectura es trivial (=1), pero se exporta por trazabilidad.
            img_to_tab = attn["img_to_tab"].detach().float().cpu().numpy()[:, :, :, 0]

            Hs, Ws = spatial.shape[-2:]
            n_tokens = tab_to_img.shape[-1]

            if Hs * Ws != n_tokens:
                raise RuntimeError(
                    f"No se puede reordenar cross-attention: "
                    f"grid={Hs}x{Ws}, tokens={n_tokens}."
                )

            tab_to_img = tab_to_img.reshape(
                tab_to_img.shape[0],
                tab_to_img.shape[1],
                Hs,
                Ws
            )
            img_to_tab = img_to_tab.reshape(
                img_to_tab.shape[0],
                img_to_tab.shape[1],
                Hs,
                Ws
            )

            ids_all.extend(batch["id"])
            preds_scaled_all.append(preds.detach().float().cpu().numpy())
            targets_scaled_all.append(targets.detach().float().cpu().numpy())
            spatial_all.append(spatial)
            tab_to_img_heads_all.append(tab_to_img)
            img_to_tab_heads_all.append(img_to_tab)

    ids_all = np.asarray(ids_all, dtype=int)
    preds_scaled_all = np.concatenate(preds_scaled_all)
    targets_scaled_all = np.concatenate(targets_scaled_all)
    spatial_all = np.concatenate(spatial_all, axis=0)
    tab_to_img_heads_all = np.concatenate(tab_to_img_heads_all, axis=0)
    img_to_tab_heads_all = np.concatenate(img_to_tab_heads_all, axis=0)

    tab_to_img_mean = tab_to_img_heads_all.mean(axis=1)
    img_to_tab_mean = img_to_tab_heads_all.mean(axis=1)

    preds_real = inverse_target_transform(preds_scaled_all, target_scaler, target_shift)
    targets_real = inverse_target_transform(targets_scaled_all, target_scaler, target_shift)
    abs_error = np.abs(preds_real - targets_real)

    Hs, Ws = spatial_all.shape[-2:]
    center_h, center_w = Hs // 2, Ws // 2

    # Entropía normalizada de tabla -> imagen: 0 = muy concentrada, 1 = uniforme.
    p = np.clip(tab_to_img_mean.reshape(len(ids_all), -1), 1e-12, 1.0)
    cross_entropy = -(p * np.log(p)).sum(axis=1) / np.log(p.shape[1])

    cross_flat_argmax = np.argmax(p, axis=1)
    cross_argmax_row = cross_flat_argmax // Ws
    cross_argmax_col = cross_flat_argmax % Ws

    summary = pd.DataFrame({
        "ID": ids_all,
        "target_real": targets_real,
        "prediction_real": preds_real,
        "absolute_error": abs_error,
        "spatial_attention_mean": spatial_all.mean(axis=(1, 2)),
        "spatial_attention_max": spatial_all.max(axis=(1, 2)),
        "spatial_attention_std": spatial_all.std(axis=(1, 2)),
        "spatial_attention_center": spatial_all[:, center_h, center_w],
        "cross_tab_to_img_center": tab_to_img_mean[:, center_h, center_w],
        "cross_tab_to_img_max": tab_to_img_mean.max(axis=(1, 2)),
        "cross_tab_to_img_entropy": cross_entropy,
        "cross_tab_to_img_argmax_row": cross_argmax_row,
        "cross_tab_to_img_argmax_col": cross_argmax_col,
    })

    # Coordenadas geográficas para relacionar interpretación con el territorio.
    coords = df_final[["ID", GEO_LAT_COL, GEO_LON_COL, PROJECTED_X_COL, PROJECTED_Y_COL]].copy()
    coords["ID"] = coords["ID"].astype(int)
    summary = coords.merge(summary, on="ID", how="right")

    npz_path = OUT_DIR / f"attention_maps_{split_name}.npz"
    csv_path = OUT_DIR / f"attention_summary_{split_name}.csv"

    np.savez_compressed(
        npz_path,
        ids=ids_all,
        spatial_attention=spatial_all.astype(np.float32),
        cross_tab_to_img_heads=tab_to_img_heads_all.astype(np.float32),
        cross_tab_to_img_mean=tab_to_img_mean.astype(np.float32),
        cross_img_to_tab_heads=img_to_tab_heads_all.astype(np.float32),
        cross_img_to_tab_mean=img_to_tab_mean.astype(np.float32),
        targets_real=targets_real.astype(np.float32),
        predictions_real=preds_real.astype(np.float32),
    )
    summary.to_csv(csv_path, index=False)

    return {
        "ids": ids_all,
        "spatial": spatial_all,
        "tab_to_img_heads": tab_to_img_heads_all,
        "tab_to_img_mean": tab_to_img_mean,
        "img_to_tab_heads": img_to_tab_heads_all,
        "img_to_tab_mean": img_to_tab_mean,
        "summary": summary,
        "npz_path": npz_path,
        "csv_path": csv_path,
    }


def plot_mean_attention(attention_export, split_name="test"):
    spatial_mean = attention_export["spatial"].mean(axis=0)
    cross_mean = attention_export["tab_to_img_mean"].mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

    im0 = axes[0].imshow(spatial_mean, cmap="viridis")
    axes[0].set_title("Spatial attention promedio")
    axes[0].set_xlabel("Columna token")
    axes[0].set_ylabel("Fila token")
    fig.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(cross_mean, cmap="magma")
    axes[1].set_title("Cross-attention tabla → imagen promedio")
    axes[1].set_xlabel("Columna token")
    axes[1].set_ylabel("Fila token")
    fig.colorbar(im1, ax=axes[1], fraction=0.046)

    fig.tight_layout()
    path = OUT_DIR / f"attention_mean_{split_name}.pdf"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    return path


def plot_representative_attention(attention_export, split_name="test"):
    summary = attention_export["summary"].reset_index(drop=True)
    errors = summary["absolute_error"].to_numpy()

    order = np.argsort(errors)
    selected = [order[0], order[len(order)//2], order[-1]]
    labels = ["Menor error", "Error mediano", "Mayor error"]

    fig, axes = plt.subplots(3, 2, figsize=(9, 12))

    for row, (idx, label) in enumerate(zip(selected, labels)):
        spatial = attention_export["spatial"][idx]
        cross = attention_export["tab_to_img_mean"][idx]
        sample_id = int(attention_export["ids"][idx])

        im0 = axes[row, 0].imshow(spatial, cmap="viridis")
        axes[row, 0].set_title(f"{label} — ID {sample_id}\nSpatial attention")
        fig.colorbar(im0, ax=axes[row, 0], fraction=0.046)

        im1 = axes[row, 1].imshow(cross, cmap="magma")
        axes[row, 1].set_title(f"{label} — ID {sample_id}\nTabla → imagen")
        fig.colorbar(im1, ax=axes[row, 1], fraction=0.046)

    fig.tight_layout()
    path = OUT_DIR / f"attention_representative_{split_name}.pdf"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    return path


def plot_cross_attention_heads(attention_export, split_name="test"):
    """
    Muestra las cabezas de cross-attention tabla -> imagen para una
    observación representativa (error mediano).
    """
    summary = attention_export["summary"].reset_index(drop=True)
    order = np.argsort(summary["absolute_error"].to_numpy())
    idx = int(order[len(order)//2])

    heads = attention_export["tab_to_img_heads"][idx]
    sample_id = int(attention_export["ids"][idx])
    n_heads = heads.shape[0]

    ncols = min(4, n_heads)
    nrows = int(np.ceil(n_heads / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 3.2*nrows), squeeze=False)

    vmin = heads.min()
    vmax = heads.max()

    for h in range(n_heads):
        r, c = divmod(h, ncols)
        im = axes[r, c].imshow(heads[h], cmap="magma", vmin=vmin, vmax=vmax)
        axes[r, c].set_title(f"Head {h+1}")
        axes[r, c].set_xlabel("Columna token")
        axes[r, c].set_ylabel("Fila token")

    for h in range(n_heads, nrows*ncols):
        r, c = divmod(h, ncols)
        axes[r, c].axis("off")

    fig.suptitle(f"Cross-attention tabla → imagen por cabeza — ID {sample_id}", y=1.02)
    fig.tight_layout()
    path = OUT_DIR / f"cross_attention_heads_{split_name}.pdf"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    return path


attention_test = collect_attention_information(
    model=model,
    loader=test_loader,
    split_name="test"
)

attention_mean_pdf = plot_mean_attention(attention_test, split_name="test")
attention_examples_pdf = plot_representative_attention(attention_test, split_name="test")
cross_attention_heads_pdf = plot_cross_attention_heads(attention_test, split_name="test")

print("Atención guardada en:")
print(attention_test["npz_path"])
print(attention_test["csv_path"])
print(attention_mean_pdf)
print(attention_examples_pdf)
print(cross_attention_heads_pdf)


In [ ]:
# EXPORTACIÓN COMPLETA 
#
# - Toda la atención del conjunto TEST
# - Solo 3 casos representativos de TRAIN:
#       menor error
#       error mediano
#       mayor error

import os
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm


# 1. Carpeta independiente para resultados del artículo

ARTICLE_DIR = OUT_DIR / "attention_article_export"

TEST_ATT_DIR = ARTICLE_DIR / "test"
TRAIN_ATT_DIR = ARTICLE_DIR / "train_representative"

TEST_ATT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_ATT_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta del artículo:")
print(ARTICLE_DIR)


# 2. Obtener predicciones de un loader SIN guardar atención

def collect_predictions_for_selection(
    model,
    loader,
    split_name="train"
):
    model.eval()

    ids_all = []
    preds_scaled_all = []
    targets_scaled_all = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc=f"Selecting representative samples: {split_name}"
        ):

            optical = batch["optical"].to(DEVICE)
            thermal = batch["thermal"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE).float()
            targets = batch["target"].to(DEVICE).float()

            # Inferencia float32
            preds = model(
                optical,
                thermal,
                tabular
            )

            ids_all.extend(
                np.asarray(batch["id"]).astype(int).tolist()
            )

            preds_scaled_all.append(
                preds.detach().float().cpu().numpy()
            )

            targets_scaled_all.append(
                targets.detach().float().cpu().numpy()
            )

    ids_all = np.asarray(
        ids_all,
        dtype=int
    )

    preds_scaled_all = np.concatenate(
        preds_scaled_all
    )

    targets_scaled_all = np.concatenate(
        targets_scaled_all
    )

    preds_real = inverse_target_transform(
        preds_scaled_all,
        target_scaler,
        target_shift
    )

    targets_real = inverse_target_transform(
        targets_scaled_all,
        target_scaler,
        target_shift
    )

    absolute_error = np.abs(
        preds_real - targets_real
    )

    result = pd.DataFrame({
        "ID": ids_all,
        "target_real": targets_real,
        "prediction_real": preds_real,
        "absolute_error": absolute_error,
    })

    return result


# 3. Seleccionar 3 casos representativos

def select_representative_samples(prediction_df):

    df = prediction_df.sort_values(
        "absolute_error"
    ).reset_index(drop=True)

    idx_low = 0

    idx_mid = len(df) // 2

    idx_high = len(df) - 1

    selected = pd.concat(
        [
            df.iloc[[idx_low]].assign(
                error_group="low_error"
            ),

            df.iloc[[idx_mid]].assign(
                error_group="median_error"
            ),

            df.iloc[[idx_high]].assign(
                error_group="high_error"
            ),
        ],
        ignore_index=True
    )

    return selected


# 4. Buscar índices de un Dataset a partir del ID

def find_dataset_indices_by_id(
    dataset,
    selected_ids
):
    selected_ids = set(
        int(x) for x in selected_ids
    )

    indices = []

    # Accedemos directamente al dataframe interno del Dataset
    # si existe.
    if hasattr(dataset, "df"):

        dataset_ids = (
            dataset.df["ID"]
            .astype(int)
            .to_numpy()
        )

        for i, sample_id in enumerate(dataset_ids):

            if int(sample_id) in selected_ids:
                indices.append(i)

    else:

        # Fallback genérico.
        for i in range(len(dataset)):

            sample = dataset[i]

            sample_id = int(
                sample["id"]
            )

            if sample_id in selected_ids:
                indices.append(i)

    return indices


# 5. Crear loader SOLO para las 3 muestras representativas

def build_representative_loader(
    base_loader,
    selected_ids
):

    dataset = base_loader.dataset

    indices = find_dataset_indices_by_id(
        dataset,
        selected_ids
    )

    if len(indices) != len(selected_ids):

        print(
            "Advertencia:",
            len(indices),
            "índices encontrados para",
            len(selected_ids),
            "IDs solicitados."
        )

    subset = Subset(
        dataset,
        indices
    )

    loader = DataLoader(
        subset,
        batch_size=min(
            3,
            len(subset)
        ),
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False
    )

    return loader


# 6. Función auxiliar para copiar los archivos generados
#    a la carpeta organizada del artículo

def copy_if_exists(
    source,
    destination_dir
):

    source = Path(source)

    if source.exists():

        destination = (
            Path(destination_dir)
            / source.name
        )

        shutil.copy2(
            source,
            destination
        )

        return destination

    return None


# 7. TEST COMPLETO

print("\n" + "="*70)
print("EXPORTANDO TODA LA ATENCIÓN DEL TEST")
print("="*70)


attention_test = collect_attention_information(
    model=model,
    loader=test_loader,
    split_name="test"
)


attention_mean_pdf = plot_mean_attention(
    attention_test,
    split_name="test"
)


attention_examples_pdf = plot_representative_attention(
    attention_test,
    split_name="test"
)


cross_attention_heads_pdf = plot_cross_attention_heads(
    attention_test,
    split_name="test"
)


# Copiar resultados TEST

test_files = [
    attention_test["npz_path"],
    attention_test["csv_path"],
    attention_mean_pdf,
    attention_examples_pdf,
    cross_attention_heads_pdf,
]


for file in test_files:

    copy_if_exists(
        file,
        TEST_ATT_DIR
    )


# 8. EVALUAR TRAIN SOLO PARA SELECCIONAR LOS 3 CASOS

print("\n" + "="*70)
print("BUSCANDO CASOS REPRESENTATIVOS DEL TRAIN")
print("="*70)


train_predictions = collect_predictions_for_selection(
    model=model,
    loader=train_eval_loader,
    split_name="train"
)


representative_train = select_representative_samples(
    train_predictions
)


print("\nMuestras seleccionadas:")

display(
    representative_train
)


# Guardar tabla
representative_csv = (
    TRAIN_ATT_DIR
    / "train_representative_samples.csv"
)

representative_train.to_csv(
    representative_csv,
    index=False
)


# También se guardan las predicciones de TODO train

train_predictions_csv = (
    TRAIN_ATT_DIR
    / "train_predictions_for_selection.csv"
)

train_predictions.to_csv(
    train_predictions_csv,
    index=False
)


# 9. CREAR LOADER SOLO PARA ESAS 3 MUESTRAS

selected_train_ids = (
    representative_train["ID"]
    .astype(int)
    .tolist()
)


train_representative_loader = (
    build_representative_loader(
        train_eval_loader,
        selected_train_ids
    )
)


print(
    "\nNúmero de muestras en loader representativo:",
    len(train_representative_loader.dataset)
)


# 10. EXTRAER ATENCIÓN SOLO PARA ESAS 3 MUESTRAS

attention_train_rep = collect_attention_information(
    model=model,
    loader=train_representative_loader,
    split_name="train_representative"
)


# Figuras

train_attention_mean_pdf = plot_mean_attention(
    attention_train_rep,
    split_name="train_representative"
)


train_attention_examples_pdf = (
    plot_representative_attention(
        attention_train_rep,
        split_name="train_representative"
    )
)


train_cross_heads_pdf = (
    plot_cross_attention_heads(
        attention_train_rep,
        split_name="train_representative"
    )
)


# Copiar resultados

train_files = [
    attention_train_rep["npz_path"],
    attention_train_rep["csv_path"],
    train_attention_mean_pdf,
    train_attention_examples_pdf,
    train_cross_heads_pdf,
]


for file in train_files:

    copy_if_exists(
        file,
        TRAIN_ATT_DIR
    )


# 11. Guardar matrices individuales de las 3 muestras TRAIN


for i in range(
    len(attention_train_rep["ids"])
):

    sample_id = int(
        attention_train_rep["ids"][i]
    )

    sample_dir = (
        TRAIN_ATT_DIR
        / f"ID_{sample_id}"
    )

    sample_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Spatial attention

    np.save(
        sample_dir
        / "spatial_attention.npy",

        attention_train_rep[
            "spatial"
        ][i].astype(np.float32)
    )


    # Cross-attention tabla -> imagen
    # promedio entre heads

    np.save(
        sample_dir
        / "cross_tab_to_img_mean.npy",

        attention_train_rep[
            "tab_to_img_mean"
        ][i].astype(np.float32)
    )


    # Cross-attention tabla -> imagen
    # por head

    np.save(
        sample_dir
        / "cross_tab_to_img_heads.npy",

        attention_train_rep[
            "tab_to_img_heads"
        ][i].astype(np.float32)
    )


    # Imagen -> tabla

    np.save(
        sample_dir
        / "cross_img_to_tab_mean.npy",

        attention_train_rep[
            "img_to_tab_mean"
        ][i].astype(np.float32)
    )


    np.save(
        sample_dir
        / "cross_img_to_tab_heads.npy",

        attention_train_rep[
            "img_to_tab_heads"
        ][i].astype(np.float32)
    )


# 12. Metadatos del experimento

metadata = {

    "device": str(DEVICE),

    "test_samples": int(
        len(attention_test["ids"])
    ),

    "train_representative_samples": 3,

    "train_representative_ids": [
        int(x)
        for x in selected_train_ids
    ],

    "selection": {
        "low_error": int(
            representative_train.loc[
                representative_train[
                    "error_group"
                ] == "low_error",
                "ID"
            ].iloc[0]
        ),

        "median_error": int(
            representative_train.loc[
                representative_train[
                    "error_group"
                ] == "median_error",
                "ID"
            ].iloc[0]
        ),

        "high_error": int(
            representative_train.loc[
                representative_train[
                    "error_group"
                ] == "high_error",
                "ID"
            ].iloc[0]
        ),
    }
}


metadata_path = (
    ARTICLE_DIR
    / "attention_export_metadata.json"
)


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4,
        ensure_ascii=False
    )


# 13. Revisar tamaño antes de crear ZIP

def get_directory_size(path):

    total = 0

    for p in Path(path).rglob("*"):

        if p.is_file():

            total += p.stat().st_size

    return total


size_bytes = get_directory_size(
    ARTICLE_DIR
)

size_mb = (
    size_bytes
    / 1024**2
)

size_gb = (
    size_bytes
    / 1024**3
)


print("\n" + "="*70)
print("TAMAÑO DE RESULTADOS")
print("="*70)

print(
    f"{size_mb:.2f} MB"
)

print(
    f"{size_gb:.3f} GB"
)


# 14. Crear ZIP

ZIP_PATH = (
    OUT_DIR
    / "attention_article_export.zip"
)


if ZIP_PATH.exists():
    ZIP_PATH.unlink()


with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zipf:

    for file_path in ARTICLE_DIR.rglob("*"):

        if file_path.is_file():

            arcname = file_path.relative_to(
                ARTICLE_DIR.parent
            )

            zipf.write(
                file_path,
                arcname=arcname
            )


# 15. Tamaño final ZIP

zip_size_mb = (
    ZIP_PATH.stat().st_size
    / 1024**2
)

zip_size_gb = (
    ZIP_PATH.stat().st_size
    / 1024**3
)


print("\n" + "="*70)
print("ZIP CREADO")
print("="*70)

print(
    "Ruta:",
    ZIP_PATH
)

print(
    f"Tamaño ZIP: {zip_size_mb:.2f} MB"
)

print(
    f"Tamaño ZIP: {zip_size_gb:.3f} GB"
)


# 16. Mostrar contenido

print("\nArchivos incluidos:\n")

for file_path in sorted(
    ARTICLE_DIR.rglob("*")
):

    if file_path.is_file():

        size = (
            file_path.stat().st_size
            / 1024**2
        )

        print(
            f"{file_path.relative_to(ARTICLE_DIR)}"
            f"  ->  {size:.2f} MB"
        )


print(
    "\n✓ Exportación terminada correctamente."
)

print(
    "✓ Toda la atención de TEST fue guardada."
)

print(
    "✓ Solo 3 casos representativos de TRAIN fueron guardados."
)

print(
    "✓ Archivo ZIP listo para descargar:"
)

print(
    ZIP_PATH
)

In [ ]:
# Variables tabulares que entran al modelo

print("=" * 60)
print("VARIABLES TABULARES DE ENTRADA AL MODELO")
print("=" * 60)

print(f"\nNúmero total de variables tabulares: {len(tabular_cols)}\n")

for i, col in enumerate(tabular_cols, start=1):
    print(f"{i:02d}. {col}")

In [ ]:
# Guardar predicciones del mejor modelo para TODO el dataset
# Train original + Validation + Test

import numpy as np
import pandas as pd

# 1. Predicciones de cada partición

pred_train_df = pd.DataFrame({
    "ID": np.asarray(ids_train, dtype=int),
    "Gradient_real": np.ravel(train_targs_real),
    "Gradient_predicted": np.ravel(train_preds_real),
    "split": "train"
})

pred_cal_df = pd.DataFrame({
    "ID": np.asarray(ids_cal, dtype=int),
    "Gradient_real": np.ravel(cal_targs_real),
    "Gradient_predicted": np.ravel(cal_preds_real),
    "split": "calibration"
})

pred_val_df = pd.DataFrame({
    "ID": np.asarray(ids_val, dtype=int),
    "Gradient_real": np.ravel(val_targs_real),
    "Gradient_predicted": np.ravel(val_preds_real),
    "split": "validation"
})

pred_test_df = pd.DataFrame({
    "ID": np.asarray(ids_test, dtype=int),
    "Gradient_real": np.ravel(test_targs_real),
    "Gradient_predicted": np.ravel(test_preds_real),
    "split": "test"
})

# 2. Unir todas las predicciones

pred_all_df = pd.concat(
    [
        pred_train_df,
        pred_cal_df,
        pred_val_df,
        pred_test_df
    ],
    ignore_index=True
)

# 3. Añadir errores

pred_all_df["Residual"] = (
    pred_all_df["Gradient_real"]
    - pred_all_df["Gradient_predicted"]
)

pred_all_df["Absolute_error"] = np.abs(
    pred_all_df["Residual"]
)

# 4. Recuperar coordenadas originales

coord_cols = [
    "ID",
    "Latitude_raw",
    "Longitude_raw",
    "x_m",
    "y_m"
]

coords_df = (
    df_final[coord_cols]
    .copy()
)

coords_df["ID"] = coords_df["ID"].astype(int)

# 5. Unir coordenadas + predicciones
predictions_full_df = coords_df.merge(
    pred_all_df,
    on="ID",
    how="inner"
)

# 6. Ordenar por ID

predictions_full_df = (
    predictions_full_df
    .sort_values("ID")
    .reset_index(drop=True)
)

# 7. Guardar

output_predictions = (
    OUT_DIR /
    "predictions_best_model_all_data.csv"
)

predictions_full_df.to_csv(
    output_predictions,
    index=False
)

print("Archivo guardado en:")
print(output_predictions)

print("\nNúmero de observaciones guardadas:")
print(len(predictions_full_df))

display(predictions_full_df.head())

In [ ]:
# Cell 13: Mapas espaciales para test

def get_colombia_base():
    url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(url)

    possible_cols = ['iso_a3', 'ISO_A3', 'ADM0_A3', 'SOV_A3']
    code_col = next(c for c in possible_cols if c in world.columns)

    colombia = world[world[code_col] == 'COL']

    minx, miny, maxx, maxy = colombia.total_bounds
    padx = (maxx - minx) * 0.05
    pady = (maxy - miny) * 0.05
    bbox = (minx - padx, maxx + padx, miny - pady, maxy + pady)
    return colombia, bbox

def base_map(ax, colombia, bbox):
    colombia.boundary.plot(ax=ax, linewidth=1.0, color='black')
    ax.set_xlim(bbox[0], bbox[1])
    ax.set_ylim(bbox[2], bbox[3])
    ax.set_aspect('auto')
    ax.grid(True, linestyle='--', linewidth=0.3)

def get_geo_cols(df):
    """
    Devuelve las columnas de coordenadas sin normalizar.
    Se usan para mapas, no para la entrada normalizada del modelo.
    """
    lon_col = GEO_LON_COL if GEO_LON_COL in df.columns else "Longitude"
    lat_col = GEO_LAT_COL if GEO_LAT_COL in df.columns else "Latitude"
    return lon_col, lat_col

def make_prediction_geodataframe(df_geo, df_pred):
    df_geo = df_geo.copy()
    df_geo["ID"] = df_geo["ID"].astype(int)

    df_pred = df_pred.copy()
    df_pred["ID"] = df_pred["ID"].astype(int)

    merged = df_geo.merge(df_pred, on="ID", how="inner")
    lon_col, lat_col = get_geo_cols(merged)

    merged = gpd.GeoDataFrame(
        merged,
        geometry=gpd.points_from_xy(merged[lon_col], merged[lat_col]),
        crs="EPSG:4326"
    ).dropna(subset=["geometry"])

    return merged

def plot_prediction_maps(gdf, prefix):
    actual_col = "actual_real"
    pred_col = "pred_real"

    gdf = gdf.copy()
    gdf["diff"] = gdf[actual_col] - gdf[pred_col]
    gdf["abs_diff"] = np.abs(gdf["diff"])

    vmin = min(gdf[actual_col].min(), gdf[pred_col].min())
    vmax = max(gdf[actual_col].max(), gdf[pred_col].max())
    vabs = np.nanmax(np.abs(gdf["diff"]))
    vabs_abs = gdf["abs_diff"].max()

    colombia, bbox = get_colombia_base()

    # Actual
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column=actual_col, cmap='jet', markersize=60, legend=True, vmin=vmin, vmax=vmax)
    fig.savefig(f"map_actual_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # Predicho
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column=pred_col, cmap='jet', markersize=60, legend=True, vmin=vmin, vmax=vmax)
    fig.savefig(f"map_predicted_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # Diferencia
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column="diff", cmap='RdBu_r', markersize=60, legend=True, vmin=-vabs, vmax=vabs)
    fig.savefig(f"map_difference_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # Diferencia absoluta
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column="abs_diff", cmap='viridis', markersize=60, legend=True, vmin=0, vmax=vabs_abs)
    fig.savefig(f"map_abs_difference_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

df_pred_test = pd.DataFrame({
    "ID": ids_test,
    "actual_real": test_targs_real,
    "pred_real": test_preds_real
})

merged_test = make_prediction_geodataframe(test_df, df_pred_test)
plot_prediction_maps(merged_test, "TEST_REAL")


In [ ]:
# Cell 14: Mapas espaciales en entrenamiento

df_pred_train = pd.DataFrame({
    "ID": ids_train,
    "actual_real": train_targs_real,
    "pred_real": train_preds_real
})

# Para mapas de entrenamiento se usa train_original_df,
# no train_df con sintéticos WSMOTER.
merged_train = make_prediction_geodataframe(train_original_df, df_pred_train)
plot_prediction_maps(merged_train, "TRAIN_REAL")


In [ ]:
# Cell 15: Mapas espaciales todo el conjunto de datos (train + val + test)

df_pred_val = pd.DataFrame({
    "ID": ids_val,
    "actual_real": val_targs_real,
    "pred_real": val_preds_real
})

# El mapa completo usa solo muestras reales.
# No se incluyen las filas sintéticas de train_df.
full_geo_df = pd.concat(
    [train_original_df, cal_df, val_df, test_df],
    ignore_index=True
).copy()

df_pred_all = pd.concat(
    [
        df_pred_train[["ID", "actual_real", "pred_real"]],
        pd.DataFrame({
            "ID": ids_cal,
            "actual_real": cal_targs_real,
            "pred_real": cal_preds_real
        })[["ID", "actual_real", "pred_real"]],
        df_pred_val[["ID", "actual_real", "pred_real"]],
        df_pred_test[["ID", "actual_real", "pred_real"]],
    ],
    ignore_index=True
)

merged_full = make_prediction_geodataframe(full_geo_df, df_pred_all)
plot_prediction_maps(merged_full, "FULL_REAL")


In [ ]:

# Cell 17: relación real vs predicho

tp = np.ravel(train_preds_real)
tt = np.ravel(train_targs_real)

cp = np.ravel(cal_preds_real)
ct = np.ravel(cal_targs_real)

vp = np.ravel(val_preds_real)
vt = np.ravel(val_targs_real)

ep = np.ravel(test_preds_real)
et = np.ravel(test_targs_real)

full_preds = np.concatenate([tp, cp, vp, ep])
full_targs = np.concatenate([tt, ct, vt, et])

datasets = {
    "TRAIN": (tp, tt),
    "CALIBRATION": (cp, ct),
    "VALIDATION": (vp, vt),
    "TEST": (ep, et),
    "FULL": (full_preds, full_targs)
}

all_vals = np.concatenate([tp, tt, cp, ct, vp, vt, ep, et])
xmin = np.nanmin(all_vals)
xmax = np.nanmax(all_vals)

pad = (xmax - xmin) * 0.06 if (xmax - xmin) > 0 else 0.5
xlim = (xmin - pad, xmax + pad)
ylim = (xmin - pad, xmax + pad)

point_color = "#1f4e5f"
line_color = "#F54927"
grid_color = "#d9d9d9"

for name, (preds, targs) in datasets.items():

    preds = np.ravel(preds)
    targs = np.ravel(targs)

    rmse = np.sqrt(mean_squared_error(targs, preds))
    mae = mean_absolute_error(targs, preds)
    r2 = r2_score(targs, preds) if len(targs) > 1 else np.nan

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.set_facecolor("#fafafa")

    ax.scatter(
        targs, preds,
        s=40,
        alpha=0.75,
        color=point_color,
        edgecolor="white",
        linewidth=0.4
    )

    ax.plot(
        [xlim[0], xlim[1]],
        [xlim[0], xlim[1]],
        linestyle="--",
        color=line_color,
        linewidth=1.6
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.set_xlabel("Gradiente real (°C/Km)", fontsize=12)
    ax.set_ylabel("Gradiente estimado (°C/Km)", fontsize=12)
    ax.set_title(f"{name}: real vs estimado", fontsize=13)

    ax.grid(True, linestyle="--", linewidth=0.6, color=grid_color)

    for spine in ax.spines.values():
        spine.set_color("#bbbbbb")
        spine.set_linewidth(0.8)

    fname = f"real_vs_estimado_{name.lower()}_REAL.pdf"
    fig.savefig(fname, dpi=300, bbox_inches='tight')

    print(
        f"{name} REAL -> RMSE={rmse:.3f}, "
        f"MAE={mae:.3f}, R2={r2:.3f}"
    )

    plt.show()


In [ ]:
# Cell 18: descargar mejor checkpoint y resultados de Optuna

import zipfile
from google.colab import files

zip_path = OUT_DIR / "best_optuna_model_and_results.zip"
files_to_zip = [
    best_ckpt_path,
    OUT_DIR / "best_hparams_optuna.json",
    OUT_DIR / "final_metrics_best_optuna.json",
    OUT_DIR / "optuna_trials.csv",
    OUT_DIR / "train_wsmoter_best_optuna.csv",
    OUT_DIR / "train_original_best_optuna.csv",
    OUT_DIR / "calibration_best_optuna.csv",
    OUT_DIR / "val_best_optuna.csv",
    OUT_DIR / "test_best_optuna.csv",
    OUT_DIR / "target_scaler_best_optuna.pkl",
    OUT_DIR / "target_shift_best_optuna.json",
    OUT_DIR / "tabular_scaler_best_optuna.pkl",
    OUT_DIR / "conformal_test_intervals.csv",
    OUT_DIR / "conformal_metrics.json",
    OUT_DIR / "conformal_calibration_scores.csv",
    OUT_DIR / "conformal_coverage_by_target_quartile.csv",
    OUT_DIR / "conformal_calibration_scores.pdf",
    OUT_DIR / "conformal_test_intervals.pdf",
    OUT_DIR / "attention_maps_test.npz",
    OUT_DIR / "attention_summary_test.csv",
    OUT_DIR / "attention_mean_test.pdf",
    OUT_DIR / "attention_representative_test.pdf",
    OUT_DIR / "cross_attention_heads_test.pdf",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        f = Path(f)
        if f.exists():
            zf.write(f, arcname=f.name)
        else:
            print("No encontrado, no se agrega al zip:", f)

print("Archivo creado:", zip_path)
files.download(str(zip_path))
